# LLM Code Cookbook — Comprehensive Interview Reference

**Purpose:** A runnable reference notebook covering every common LLM operation — from similarity metrics to RAG pipelines to fine-tuning. Each section is self-contained with from-scratch implementations AND library usage.

**How to use:** Read the markdown explanations, then run each code cell. Toy/synthetic data is used throughout so nothing requires a GPU or large downloads.

---

In [ ]:
# Run this cell first to install/import common dependencies
# !pip install numpy torch scikit-learn matplotlib seaborn tqdm

import numpy as np
import warnings
warnings.filterwarnings('ignore')
print("Common imports ready.")

---
# Section 1: Similarity & Distance Metrics

Understanding distance/similarity is fundamental to embeddings, retrieval, and clustering.
Key concepts:
- **Cosine similarity**: measures angle between vectors (range [-1,1]). Ignores magnitude.
- **Dot product**: measures both direction AND magnitude. Used in attention.
- **Euclidean distance**: L2 distance. Affected by magnitude.

**Interview tip:** Know when to use each. Cosine is standard for normalized embeddings. Dot product is used when magnitude matters (e.g., attention scores). Euclidean is used in k-means.

### 1a. Cosine Similarity from Scratch (NumPy)

In [ ]:
import numpy as np

def cosine_similarity_scratch(a, b):
    """Compute cosine similarity between two vectors."""
    dot = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

# Example
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
c = np.array([-1.0, -2.0, -3.0])  # opposite direction to a

print(f"cos(a, b) = {cosine_similarity_scratch(a, b):.4f}")   # ~0.9746
print(f"cos(a, c) = {cosine_similarity_scratch(a, c):.4f}")   # -1.0 (opposite)
print(f"cos(a, a) = {cosine_similarity_scratch(a, a):.4f}")   # 1.0 (identical)

### 1b. Pairwise Cosine Similarity Matrix

In [ ]:
import numpy as np

def pairwise_cosine_similarity(X):
    """Compute NxN cosine similarity matrix for N vectors.
    X: shape (N, D)
    """
    # Normalize each row to unit length
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)  # avoid division by zero
    X_norm = X / norms
    # Pairwise cosine sim = dot product of normalized vectors
    return X_norm @ X_norm.T

# Example: 4 vectors of dimension 3
X = np.array([
    [1, 0, 0],
    [0, 1, 0],
    [1, 1, 0],
    [-1, 0, 0]
], dtype=float)

sim_matrix = pairwise_cosine_similarity(X)
print("Pairwise cosine similarity matrix:")
print(np.round(sim_matrix, 3))
# Diagonal should be 1.0, X[0] and X[3] should be -1.0 (opposite)

### 1c. Dot Product Similarity

In [ ]:
import numpy as np

def dot_product_similarity(a, b):
    """Dot product — used in attention and when magnitude matters."""
    return np.dot(a, b)

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(f"dot(a, b) = {dot_product_similarity(a, b)}")  # 32.0

# Note: for normalized vectors, dot product == cosine similarity
a_norm = a / np.linalg.norm(a)
b_norm = b / np.linalg.norm(b)
print(f"dot(a_norm, b_norm) = {dot_product_similarity(a_norm, b_norm):.4f}")
print(f"cosine(a, b) = {cosine_similarity_scratch(a, b):.4f}")
# These two should be equal

### 1d. Euclidean Distance

In [ ]:
import numpy as np

def euclidean_distance(a, b):
    """L2 distance between two vectors."""
    return np.sqrt(np.sum((a - b) ** 2))

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0, 6.0])
print(f"L2(a, b) = {euclidean_distance(a, b):.4f}")  # 5.196

# Relationship: for unit vectors, L2^2 = 2 - 2*cos(a,b)
a_n = a / np.linalg.norm(a)
b_n = b / np.linalg.norm(b)
l2_sq = euclidean_distance(a_n, b_n) ** 2
cos_val = cosine_similarity_scratch(a, b)
print(f"L2^2 = {l2_sq:.4f}, 2 - 2*cos = {2 - 2*cos_val:.4f}")  # should match

### 1e. Using sklearn and torch for Similarity/Distance

In [ ]:
import numpy as np

# --- sklearn ---
from sklearn.metrics.pairwise import cosine_similarity as sk_cos
from sklearn.metrics.pairwise import euclidean_distances as sk_euc

X = np.array([[1,0,0],[0,1,0],[1,1,0]], dtype=float)
print("sklearn cosine similarity matrix:")
print(np.round(sk_cos(X), 3))

print("\nsklearn euclidean distance matrix:")
print(np.round(sk_euc(X), 3))

# --- torch ---
import torch
X_t = torch.tensor(X, dtype=torch.float32)

# Cosine similarity (pairwise) via normalize + matmul
X_norm = torch.nn.functional.normalize(X_t, dim=1)
cos_mat = X_norm @ X_norm.T
print("\ntorch cosine similarity matrix:")
print(cos_mat.numpy().round(3))

# Euclidean distance
euc_mat = torch.cdist(X_t, X_t, p=2)
print("\ntorch euclidean distance matrix:")
print(euc_mat.numpy().round(3))

### 1f. When to Use Which Metric — Comparison

| Metric | Range | Sensitive to Magnitude? | Common Use |
|--------|-------|------------------------|------------|
| Cosine | [-1, 1] | No | Embedding similarity, retrieval |
| Dot Product | (-inf, inf) | Yes | Attention scores, when magnitude = importance |
| Euclidean | [0, inf) | Yes | Clustering (k-means), kNN |

**Key insight:** If embeddings are L2-normalized, cosine similarity = dot product, and ranking by Euclidean distance gives the same order as ranking by cosine similarity.

In [ ]:
# Demonstration: all three metrics agree on ranking when vectors are normalized
import numpy as np

query = np.random.randn(128)
docs = np.random.randn(5, 128)

# Normalize everything
query_n = query / np.linalg.norm(query)
docs_n = docs / np.linalg.norm(docs, axis=1, keepdims=True)

cos_scores = docs_n @ query_n
dot_scores = docs_n @ query_n  # same as cosine when normalized
euc_dists = np.linalg.norm(docs_n - query_n, axis=1)

print("Rankings (should be identical for normalized vectors):")
print(f"  Cosine (desc):    {np.argsort(-cos_scores)}")
print(f"  Dot product (desc): {np.argsort(-dot_scores)}")
print(f"  Euclidean (asc):  {np.argsort(euc_dists)}")

---
# Section 2: Tokenization

Tokenization converts text into integer IDs. Modern LLMs use subword tokenization (BPE, WordPiece, Unigram).

**Key concepts:**
- **BPE (Byte Pair Encoding):** Iteratively merges the most frequent pair of tokens.
- **WordPiece:** Similar to BPE but uses likelihood-based merging (used by BERT).
- **SentencePiece/Unigram:** Probabilistic model over subwords (used by T5, LLaMA).

**Interview tip:** Know how BPE works step-by-step, how vocab size affects model performance, and how to count tokens for context window management.

### 2a. BPE Tokenization from Scratch (Simplified)

In [ ]:
import re
from collections import Counter

class SimpleBPE:
    """Simplified BPE tokenizer for educational purposes."""

    def __init__(self, vocab_size=300):
        self.vocab_size = vocab_size
        self.merges = []  # list of (pair, merged_token)

    def _get_pairs(self, tokens_list):
        """Count frequency of adjacent token pairs across all words."""
        pairs = Counter()
        for tokens in tokens_list:
            for i in range(len(tokens) - 1):
                pairs[(tokens[i], tokens[i+1])] += 1
        return pairs

    def _merge_pair(self, tokens_list, pair):
        """Merge all occurrences of pair in every word."""
        merged = pair[0] + pair[1]
        new_tokens_list = []
        for tokens in tokens_list:
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            new_tokens_list.append(new_tokens)
        return new_tokens_list

    def train(self, corpus):
        """Train BPE on a corpus (list of strings)."""
        # Initialize: split each word into characters
        words = []
        for text in corpus:
            for word in text.lower().split():
                words.append(list(word) + ['</w>'])  # end-of-word marker

        # Get initial vocab (all characters)
        vocab = set()
        for w in words:
            vocab.update(w)
        print(f"Initial vocab size: {len(vocab)}")

        # Iteratively merge most frequent pair
        while len(vocab) < self.vocab_size:
            pairs = self._get_pairs(words)
            if not pairs:
                break
            best_pair = pairs.most_common(1)[0][0]
            merged_token = best_pair[0] + best_pair[1]
            self.merges.append(best_pair)
            vocab.add(merged_token)
            words = self._merge_pair(words, best_pair)

        print(f"Final vocab size: {len(vocab)}, Merges learned: {len(self.merges)}")
        return self.merges

    def tokenize(self, word):
        """Tokenize a single word using learned merges."""
        tokens = list(word.lower()) + ['</w>']
        for pair in self.merges:
            merged = pair[0] + pair[1]
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                    new_tokens.append(merged)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens
        return tokens

# Train on sample corpus
corpus = [
    "the cat sat on the mat",
    "the cat ate the food",
    "the dog sat on the log",
    "a cat and a dog",
] * 10  # repeat to get more frequency signal

bpe = SimpleBPE(vocab_size=50)
merges = bpe.train(corpus)
print(f"\nFirst 10 merges: {merges[:10]}")

# Tokenize a new word
for word in ["cat", "the", "eating", "dogs"]:
    print(f"  '{word}' -> {bpe.tokenize(word)}")

### 2b. HuggingFace Tokenizer Usage (AutoTokenizer)

In [ ]:
# NOTE: requires `pip install transformers`
# Using a small model tokenizer that downloads quickly
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

    text = "Tokenization is fundamental to language models."
    encoded = tokenizer(text)
    print(f"Input: {text}")
    print(f"Token IDs: {encoded['input_ids']}")
    print(f"Tokens: {tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")
    print(f"Num tokens: {len(encoded['input_ids'])}")
    print(f"Vocab size: {tokenizer.vocab_size}")
except ImportError:
    print("Install transformers: pip install transformers")

### 2c. Token Count Estimation

In [ ]:
def estimate_token_count(text, method="words"):
    """Quick token count estimates without loading a tokenizer.
    Rules of thumb:
    - English: ~1.3 tokens per word (for GPT-style BPE)
    - Characters: ~4 chars per token
    """
    if method == "words":
        word_count = len(text.split())
        return int(word_count * 1.3)
    elif method == "chars":
        return len(text) // 4
    else:
        raise ValueError("method must be 'words' or 'chars'")

text = "The quick brown fox jumps over the lazy dog. This is a test sentence for estimation."
print(f"Text: {text}")
print(f"Word count: {len(text.split())}")
print(f"Estimated tokens (word method): {estimate_token_count(text, 'words')}")
print(f"Estimated tokens (char method): {estimate_token_count(text, 'chars')}")

# Compare with actual tokenizer if available
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    actual = len(tok.encode(text))
    print(f"Actual tokens (BERT): {actual}")
except:
    pass

### 2d. Vocab Overlap Analysis Between Two Tokenizers

In [ ]:
try:
    from transformers import AutoTokenizer

    tok1 = AutoTokenizer.from_pretrained("bert-base-uncased")
    tok2 = AutoTokenizer.from_pretrained("gpt2")

    vocab1 = set(tok1.get_vocab().keys())
    vocab2 = set(tok2.get_vocab().keys())

    overlap = vocab1 & vocab2
    print(f"BERT vocab size: {len(vocab1)}")
    print(f"GPT-2 vocab size: {len(vocab2)}")
    print(f"Overlap: {len(overlap)} tokens")
    print(f"Overlap %: {100*len(overlap)/min(len(vocab1),len(vocab2)):.1f}%")
    print(f"Sample overlapping tokens: {list(overlap)[:20]}")
except ImportError:
    print("Install transformers: pip install transformers")

### 2e. Decode Tokens Back to Text, Inspect Subwords

In [ ]:
try:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    text = "unbelievably antidisestablishmentarianism"
    ids = tokenizer.encode(text)
    tokens = tokenizer.convert_ids_to_tokens(ids)

    print(f"Text: {text}")
    print(f"Tokens: {tokens}")
    print(f"IDs: {ids}")
    # Decode back
    decoded = tokenizer.decode(ids, skip_special_tokens=True)
    print(f"Decoded: {decoded}")

    # Show which tokens are subwords (start with ##)
    for tok in tokens:
        if tok.startswith("##"):
            print(f"  Subword continuation: {tok}")
except ImportError:
    print("Install transformers: pip install transformers")

---
# Section 3: Attention Mechanism

Attention is the core of the Transformer. It computes a weighted sum of values, where weights are determined by query-key compatibility.

**Formula:** Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) V

**Interview tip:** Be ready to implement scaled dot-product attention from scratch, explain why we scale by sqrt(d_k), and explain causal masking for autoregressive models.

### 3a. Scaled Dot-Product Attention from Scratch (NumPy)

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (seq_len_q, d_k)
    K: (seq_len_k, d_k)
    V: (seq_len_k, d_v)
    mask: (seq_len_q, seq_len_k) — True means MASK (set to -inf)
    Returns: output (seq_len_q, d_v), attention_weights (seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]

    # Step 1: QK^T — compute attention scores
    scores = Q @ K.T  # (seq_len_q, seq_len_k)

    # Step 2: Scale by sqrt(d_k) to prevent softmax saturation
    # Without scaling, large d_k -> large dot products -> peaked softmax -> vanishing gradients
    scores = scores / np.sqrt(d_k)

    # Step 3: Apply mask (for causal / padding)
    if mask is not None:
        scores = np.where(mask, -1e9, scores)

    # Step 4: Softmax to get attention weights
    attn_weights = softmax(scores, axis=-1)

    # Step 5: Weighted sum of values
    output = attn_weights @ V  # (seq_len_q, d_v)

    return output, attn_weights

# Example: 4 tokens, dimension 8
np.random.seed(42)
seq_len, d_k, d_v = 4, 8, 8
Q = np.random.randn(seq_len, d_k)
K = np.random.randn(seq_len, d_k)
V = np.random.randn(seq_len, d_v)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")      # (4, 8)
print(f"Attention weights shape: {weights.shape}")  # (4, 4)
print(f"Attention weights (rows sum to 1):")
print(np.round(weights, 3))
print(f"Row sums: {weights.sum(axis=-1).round(4)}")  # all 1.0

### 3b. Causal (Masked) Attention from Scratch

In [ ]:
import numpy as np

def causal_attention(Q, K, V):
    """Attention with causal mask — token i can only attend to tokens <= i.
    This is used in decoder / autoregressive models (GPT, LLaMA).
    """
    seq_len = Q.shape[0]
    # Create causal mask: upper triangle is True (masked)
    # Position (i, j): True if j > i (future token)
    causal_mask = np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)
    return scaled_dot_product_attention(Q, K, V, mask=causal_mask)

# Reuse Q, K, V from above
output, weights = causal_attention(Q, K, V)
print("Causal attention weights:")
print(np.round(weights, 3))
# Notice: upper triangle should be ~0 (masked out)
# Row 0: only attends to itself
# Row 1: attends to 0 and 1
# Row 3: attends to 0, 1, 2, 3

### 3c. Multi-Head Attention from Scratch (PyTorch)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """Multi-head attention from scratch.

    Key idea: instead of one big attention, split into h heads.
    Each head learns different attention patterns.
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # dimension per head

        # Linear projections for Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # output projection

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        seq_len_q = query.size(1)
        seq_len_k = key.size(1)

        # 1. Linear projections: (batch, seq, d_model) -> (batch, seq, d_model)
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 2. Reshape to (batch, num_heads, seq, d_k)
        Q = Q.view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)

        # 3. Scaled dot-product attention per head
        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)  # (batch, heads, seq_q, seq_k)

        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)  # (batch, heads, seq_q, seq_k)
        attn_output = attn_weights @ V             # (batch, heads, seq_q, d_k)

        # 4. Concatenate heads: (batch, seq_q, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len_q, self.d_model)

        # 5. Final linear projection
        output = self.W_o(attn_output)
        return output, attn_weights

# Test
batch, seq_len, d_model, num_heads = 2, 6, 64, 8
mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch, seq_len, d_model)
output, weights = mha(x, x, x)  # self-attention
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")          # (2, 6, 64)
print(f"Attention weights shape: {weights.shape}")  # (2, 8, 6, 6)

### 3d. Visualize Attention Weights with Heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generate sample attention weights
np.random.seed(0)
Q = np.random.randn(6, 16)
K = np.random.randn(6, 16)
V = np.random.randn(6, 16)

_, attn_weights = scaled_dot_product_attention(Q, K, V)
tokens = ["The", "cat", "sat", "on", "the", "mat"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bidirectional attention
axes[0].imshow(attn_weights, cmap='Blues', vmin=0, vmax=1)
axes[0].set_xticks(range(len(tokens))); axes[0].set_xticklabels(tokens)
axes[0].set_yticks(range(len(tokens))); axes[0].set_yticklabels(tokens)
axes[0].set_title("Bidirectional Attention")
axes[0].set_xlabel("Key"); axes[0].set_ylabel("Query")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        axes[0].text(j, i, f"{attn_weights[i,j]:.2f}", ha='center', va='center', fontsize=8)

# Causal attention
_, causal_weights = causal_attention(Q, K, V)
axes[1].imshow(causal_weights, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks(range(len(tokens))); axes[1].set_xticklabels(tokens)
axes[1].set_yticks(range(len(tokens))); axes[1].set_yticklabels(tokens)
axes[1].set_title("Causal (Masked) Attention")
axes[1].set_xlabel("Key"); axes[1].set_ylabel("Query")
for i in range(len(tokens)):
    for j in range(len(tokens)):
        axes[1].text(j, i, f"{causal_weights[i,j]:.2f}", ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=100, bbox_inches='tight')
plt.show()
print("Saved attention_heatmap.png")

### 3e. Count FLOPs for Attention Computation

In [ ]:
def attention_flops(seq_len, d_model, num_heads):
    """Estimate FLOPs for multi-head self-attention.

    Main operations:
    1. Q, K, V projections: 3 * (2 * seq_len * d_model * d_model) = 6 * seq_len * d_model^2
    2. QK^T: 2 * seq_len^2 * d_model (per head, summed across heads)
    3. softmax: ~5 * seq_len^2 * num_heads (approximate)
    4. attn @ V: 2 * seq_len^2 * d_model
    5. output projection: 2 * seq_len * d_model^2

    Total ≈ 8 * seq_len * d_model^2 + 4 * seq_len^2 * d_model
    """
    projection_flops = 8 * seq_len * d_model ** 2  # Q,K,V + output projections
    attention_flops_val = 4 * seq_len ** 2 * d_model  # QK^T + attn@V (across all heads)
    total = projection_flops + attention_flops_val
    return {
        "projection_flops": projection_flops,
        "attention_matrix_flops": attention_flops_val,
        "total_flops": total,
        "ratio_quadratic": attention_flops_val / total
    }

# Compare different sequence lengths
for seq_len in [512, 2048, 8192, 32768]:
    result = attention_flops(seq_len, d_model=4096, num_heads=32)
    print(f"seq_len={seq_len:>6}: total={result['total_flops']:.2e} FLOPs, "
          f"quadratic portion={result['ratio_quadratic']:.1%}")
# Note: as seq_len grows, the quadratic O(n^2) portion dominates

---
# Section 4: Transformer Components

Building blocks of the Transformer: normalization, feed-forward networks, positional encodings.

**Interview tip:** Know LayerNorm vs RMSNorm (modern LLMs use RMSNorm). Know pre-norm vs post-norm. Know RoPE (used by LLaMA, Mistral).

### 4a. Layer Normalization from Scratch

In [ ]:
import numpy as np

def layer_norm(x, gamma, beta, eps=1e-5):
    """Layer Normalization: normalize across the feature dimension.
    x: (batch, ..., d_model)
    gamma, beta: learnable scale and shift, shape (d_model,)

    Unlike BatchNorm, LayerNorm normalizes across features (not batch).
    This makes it suitable for variable-length sequences and small batches.
    """
    mean = np.mean(x, axis=-1, keepdims=True)
    var = np.var(x, axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    return gamma * x_norm + beta

# Test
np.random.seed(42)
x = np.random.randn(2, 4, 8)  # (batch=2, seq=4, d_model=8)
gamma = np.ones(8)
beta = np.zeros(8)

out = layer_norm(x, gamma, beta)
print(f"Input mean per token: {x[0, 0].mean():.4f}, var: {x[0, 0].var():.4f}")
print(f"Output mean per token: {out[0, 0].mean():.6f}, var: {out[0, 0].var():.4f}")
# Output should have ~0 mean and ~1 variance per token

### 4b. RMSNorm from Scratch

In [ ]:
import numpy as np

def rms_norm(x, gamma, eps=1e-6):
    """Root Mean Square Layer Normalization.
    Used in LLaMA, Mistral, etc. instead of LayerNorm.
    Simpler than LayerNorm: no mean centering, no beta.
    Slightly faster and works just as well.

    RMS(x) = sqrt(mean(x^2))
    RMSNorm(x) = x / RMS(x) * gamma
    """
    rms = np.sqrt(np.mean(x ** 2, axis=-1, keepdims=True) + eps)
    return (x / rms) * gamma

# Compare LayerNorm vs RMSNorm
x = np.random.randn(2, 4, 8)
gamma = np.ones(8)
beta = np.zeros(8)

ln_out = layer_norm(x, gamma, beta)
rms_out = rms_norm(x, gamma)
print(f"LayerNorm output[0,0]: {ln_out[0,0].round(3)}")
print(f"RMSNorm  output[0,0]: {rms_out[0,0].round(3)}")
# Similar but not identical — RMSNorm doesn't center to zero mean

### 4c. Feed-Forward Network (with ReLU and SwiGLU)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FFN_ReLU(nn.Module):
    """Standard FFN: Linear -> ReLU -> Linear (used in original Transformer)."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))

class FFN_SwiGLU(nn.Module):
    """SwiGLU FFN: used in LLaMA, PaLM, Mistral.
    SwiGLU(x) = (Swish(xW1) * xV) W2
    Has 3 weight matrices instead of 2, but d_ff is reduced to compensate.
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)  # gate
        self.v = nn.Linear(d_model, d_ff, bias=False)    # value
        self.w2 = nn.Linear(d_ff, d_model, bias=False)   # down

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))  # SiLU = Swish

# Compare parameter counts
d_model, d_ff = 512, 2048
ffn_relu = FFN_ReLU(d_model, d_ff)
ffn_swiglu = FFN_SwiGLU(d_model, d_ff)

relu_params = sum(p.numel() for p in ffn_relu.parameters())
swiglu_params = sum(p.numel() for p in ffn_swiglu.parameters())
print(f"FFN_ReLU params: {relu_params:,}")     # 2 * d_model * d_ff + biases
print(f"FFN_SwiGLU params: {swiglu_params:,}")  # 3 * d_model * d_ff (no bias)

x = torch.randn(2, 10, d_model)
print(f"FFN_ReLU output: {ffn_relu(x).shape}")
print(f"FFN_SwiGLU output: {ffn_swiglu(x).shape}")

### 4d. Full Transformer Encoder Layer from Scratch

In [ ]:
import torch
import torch.nn as nn

class TransformerEncoderLayer(nn.Module):
    """Full encoder layer with pre-norm (modern style).
    Architecture: x -> Norm -> MHA -> Add -> Norm -> FFN -> Add
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-norm + self-attention + residual
        x_norm = self.norm1(x)
        attn_out, _ = self.mha(x_norm, x_norm, x_norm, attn_mask=mask)
        x = x + self.dropout(attn_out)

        # Pre-norm + FFN + residual
        x_norm = self.norm2(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout(ffn_out)
        return x

# Test
d_model, num_heads, d_ff = 64, 4, 256
encoder_layer = TransformerEncoderLayer(d_model, num_heads, d_ff)
x = torch.randn(2, 10, d_model)  # (batch=2, seq=10, d=64)
out = encoder_layer(x)
print(f"Encoder input: {x.shape}, output: {out.shape}")
print(f"Params: {sum(p.numel() for p in encoder_layer.parameters()):,}")

### 4e. Full Transformer Decoder Layer (with Causal Mask) from Scratch

In [ ]:
import torch
import torch.nn as nn

class TransformerDecoderLayer(nn.Module):
    """Decoder layer with causal self-attention + optional cross-attention.
    For GPT-style (decoder-only), skip cross-attention.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def generate_causal_mask(seq_len):
        """Upper triangular mask: True = masked positions."""
        return torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

    def forward(self, x, encoder_output=None):
        seq_len = x.size(1)
        causal_mask = self.generate_causal_mask(seq_len).to(x.device)

        # Causal self-attention
        x_norm = self.norm1(x)
        attn_out, _ = self.self_attn(x_norm, x_norm, x_norm, attn_mask=causal_mask)
        x = x + self.dropout(attn_out)

        # Cross-attention (if encoder output provided — for seq2seq)
        if encoder_output is not None:
            x_norm = self.norm2(x)
            cross_out, _ = self.cross_attn(x_norm, encoder_output, encoder_output)
            x = x + self.dropout(cross_out)

        # FFN
        x_norm = self.norm3(x)
        ffn_out = self.ffn(x_norm)
        x = x + self.dropout(ffn_out)
        return x

# Test decoder-only (no cross attention)
d_model, num_heads, d_ff = 64, 4, 256
decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff)
x = torch.randn(2, 10, d_model)
out = decoder_layer(x)
print(f"Decoder-only: input {x.shape} -> output {out.shape}")

# Test with cross-attention (encoder-decoder)
enc_out = torch.randn(2, 20, d_model)
out = decoder_layer(x, encoder_output=enc_out)
print(f"Enc-Dec: decoder input {x.shape}, encoder {enc_out.shape} -> output {out.shape}")

### 4f. Positional Encoding (Sinusoidal) from Scratch + Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_positional_encoding(max_len, d_model):
    """Sinusoidal positional encoding from 'Attention Is All You Need'.
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

    Properties:
    - Deterministic (no learned parameters)
    - Can extrapolate to longer sequences than seen during training
    - PE(pos+k) can be represented as a linear function of PE(pos)
    """
    pe = np.zeros((max_len, d_model))
    position = np.arange(max_len)[:, np.newaxis]  # (max_len, 1)
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

    pe[:, 0::2] = np.sin(position * div_term)  # even indices
    pe[:, 1::2] = np.cos(position * div_term)  # odd indices
    return pe

pe = sinusoidal_positional_encoding(100, 64)
print(f"PE shape: {pe.shape}")  # (100, 64)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(pe, aspect='auto', cmap='RdBu')
axes[0].set_xlabel("Dimension"); axes[0].set_ylabel("Position")
axes[0].set_title("Sinusoidal Positional Encoding")

# Show that nearby positions have similar encodings
from sklearn.metrics.pairwise import cosine_similarity as sk_cos
pe_sim = sk_cos(pe[:20])
axes[1].imshow(pe_sim, cmap='Blues')
axes[1].set_title("Cosine Sim of Positions 0-19")
axes[1].set_xlabel("Position"); axes[1].set_ylabel("Position")
plt.tight_layout()
plt.savefig("positional_encoding.png", dpi=100, bbox_inches='tight')
plt.show()

### 4g. Rotary Position Embedding (RoPE) from Scratch

In [ ]:
import torch
import numpy as np

def precompute_rope_freqs(d_model, max_len=1024, theta=10000.0):
    """Precompute RoPE frequency tensor.
    RoPE encodes position by rotating pairs of dimensions.
    Used in LLaMA, Mistral, and most modern LLMs.

    Key property: dot product between RoPE-encoded vectors depends on
    relative position, not absolute position.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, d_model, 2).float() / d_model))
    t = torch.arange(max_len)
    freqs = torch.outer(t, freqs)  # (max_len, d_model/2)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos_freqs, sin_freqs):
    """Apply RoPE to input tensor x of shape (batch, seq_len, d_model).
    Rotates pairs of dimensions: (x0, x1) -> (x0*cos - x1*sin, x0*sin + x1*cos)
    """
    seq_len = x.shape[1]
    cos_f = cos_freqs[:seq_len].unsqueeze(0)  # (1, seq_len, d/2)
    sin_f = sin_freqs[:seq_len].unsqueeze(0)

    # Split into pairs
    x1 = x[..., 0::2]  # even dimensions
    x2 = x[..., 1::2]  # odd dimensions

    # Apply rotation
    out1 = x1 * cos_f - x2 * sin_f
    out2 = x1 * sin_f + x2 * cos_f

    # Interleave back
    out = torch.stack([out1, out2], dim=-1).flatten(-2)
    return out

# Test
d_model = 64
cos_f, sin_f = precompute_rope_freqs(d_model, max_len=128)
x = torch.randn(1, 10, d_model)

x_rope = apply_rope(x, cos_f, sin_f)
print(f"Input shape: {x.shape}, RoPE output: {x_rope.shape}")

# Verify relative position property
# Dot product of position i and j should depend on |i-j|, not i or j
q = torch.randn(1, 20, d_model)
k = torch.randn(1, 20, d_model)
q_rope = apply_rope(q, cos_f, sin_f)
k_rope = apply_rope(k, cos_f, sin_f)

# Compare dot(q[5], k[3]) vs dot(q[10], k[8]) — both have relative distance 2
dot1 = (q_rope[0, 5] * k_rope[0, 3]).sum().item()
dot2 = (q_rope[0, 10] * k_rope[0, 8]).sum().item()
print(f"dot(pos5, pos3) = {dot1:.4f}")
print(f"dot(pos10, pos8) = {dot2:.4f}")
print("(These should be similar — same relative distance)")

---
# Section 5: Model Inspection & Parameter Counting

Understanding model architecture and sizing is critical for deployment planning.

**Interview tip:** Be able to estimate memory requirements for any model given its parameter count. Know the relationship: params * bytes_per_param = memory.

### 5a. Load a HuggingFace Model and Print Architecture

In [ ]:
try:
    from transformers import AutoModel, AutoConfig

    # Load just the config (no weights download) for inspection
    config = AutoConfig.from_pretrained("bert-base-uncased")
    print("BERT-base config:")
    print(f"  Hidden size: {config.hidden_size}")
    print(f"  Num layers: {config.num_hidden_layers}")
    print(f"  Num heads: {config.num_attention_heads}")
    print(f"  Intermediate (FFN) size: {config.intermediate_size}")
    print(f"  Vocab size: {config.vocab_size}")
    print(f"  Max position: {config.max_position_embeddings}")

    # Load actual model to see full architecture
    model = AutoModel.from_pretrained("bert-base-uncased")
    print(f"\nFull architecture:\n{model}")
except ImportError:
    print("Install transformers: pip install transformers")

### 5b. Count Total Parameters, Trainable Parameters

In [ ]:
def count_parameters(model):
    """Count total and trainable parameters in a PyTorch model."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen = total - trainable
    return {"total": total, "trainable": trainable, "frozen": frozen}

try:
    from transformers import AutoModel
    model = AutoModel.from_pretrained("bert-base-uncased")
    counts = count_parameters(model)
    print(f"BERT-base parameters:")
    print(f"  Total:     {counts['total']:>12,}")
    print(f"  Trainable: {counts['trainable']:>12,}")
    print(f"  Frozen:    {counts['frozen']:>12,}")
except ImportError:
    # Fallback: use a simple torch model
    import torch.nn as nn
    model = nn.Sequential(nn.Linear(768, 3072), nn.ReLU(), nn.Linear(3072, 768))
    counts = count_parameters(model)
    print(f"Simple FFN: {counts}")

### 5c. Parameter Count per Layer Type

In [ ]:
def params_by_layer_type(model):
    """Break down parameters by layer type (attention, FFN, embedding, norm)."""
    breakdown = {}
    for name, param in model.named_parameters():
        # Categorize by name patterns
        if 'embed' in name:
            category = 'embedding'
        elif 'attention' in name or 'attn' in name:
            category = 'attention'
        elif 'intermediate' in name or 'ffn' in name or 'mlp' in name:
            category = 'ffn'
        elif 'norm' in name or 'LayerNorm' in name:
            category = 'norm'
        elif 'pooler' in name:
            category = 'pooler'
        else:
            category = 'other'
        breakdown[category] = breakdown.get(category, 0) + param.numel()
    return breakdown

try:
    from transformers import AutoModel
    model = AutoModel.from_pretrained("bert-base-uncased")
    breakdown = params_by_layer_type(model)
    total = sum(breakdown.values())
    print("Parameter breakdown by type:")
    for cat, count in sorted(breakdown.items(), key=lambda x: -x[1]):
        print(f"  {cat:>12}: {count:>12,}  ({100*count/total:.1f}%)")
except ImportError:
    print("Install transformers: pip install transformers")

### 5d. Estimate Model Memory (FP32, FP16, INT8, INT4)

In [ ]:
def estimate_model_memory(num_params, dtype="fp16"):
    """Estimate memory in GB for a model with given parameter count.

    Bytes per parameter:
    - FP32: 4 bytes
    - FP16/BF16: 2 bytes
    - INT8: 1 byte
    - INT4: 0.5 bytes

    For training, multiply by ~4x (optimizer states, gradients, activations).
    For inference, just model weights + KV cache + activations.
    """
    bytes_per_param = {
        "fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5
    }
    bpp = bytes_per_param.get(dtype, 2)
    memory_bytes = num_params * bpp
    memory_gb = memory_bytes / (1024 ** 3)
    return memory_gb

# Common model sizes
models = {
    "BERT-base": 110_000_000,
    "BERT-large": 340_000_000,
    "GPT-2": 124_000_000,
    "GPT-2 XL": 1_500_000_000,
    "LLaMA-7B": 7_000_000_000,
    "LLaMA-13B": 13_000_000_000,
    "LLaMA-70B": 70_000_000_000,
}

print(f"{'Model':<15} {'Params':<12} {'FP32':>8} {'FP16':>8} {'INT8':>8} {'INT4':>8}")
print("-" * 70)
for name, params in models.items():
    row = f"{name:<15} {params/1e9:.1f}B"
    for dtype in ["fp32", "fp16", "int8", "int4"]:
        mem = estimate_model_memory(params, dtype)
        row += f"  {mem:>6.1f}GB"
    print(row)

### 5e. Estimate KV Cache Size for a Given Sequence Length

In [ ]:
def kv_cache_size_gb(num_layers, d_model, max_seq_len, batch_size=1,
                     dtype_bytes=2, num_kv_heads=None, num_heads=32):
    """Estimate KV cache memory.

    KV cache stores key and value tensors for all past tokens across all layers.
    Shape per layer: 2 (K+V) * batch * seq_len * d_head * num_kv_heads

    For GQA (Grouped Query Attention): num_kv_heads < num_heads
    For MHA: num_kv_heads = num_heads
    """
    if num_kv_heads is None:
        num_kv_heads = num_heads  # standard MHA

    d_head = d_model // num_heads
    # 2 for K and V, per layer
    total_elements = 2 * num_layers * batch_size * max_seq_len * d_head * num_kv_heads
    total_bytes = total_elements * dtype_bytes
    return total_bytes / (1024 ** 3)

print("KV Cache Size Estimates (FP16, batch=1):")
print(f"{'Config':<30} {'Seq 2K':>10} {'Seq 8K':>10} {'Seq 32K':>10} {'Seq 128K':>10}")
print("-" * 75)

configs = [
    ("LLaMA-7B (MHA)", 32, 4096, 32, 32),
    ("LLaMA-7B (GQA, 8 KV)", 32, 4096, 32, 8),
    ("LLaMA-70B (GQA, 8 KV)", 80, 8192, 64, 8),
]

for name, nl, dm, nh, nkv in configs:
    row = f"{name:<30}"
    for seq in [2048, 8192, 32768, 131072]:
        mem = kv_cache_size_gb(nl, dm, seq, 1, 2, nkv, nh)
        row += f"  {mem:>8.2f}GB"
    print(row)

print("\nNote: GQA significantly reduces KV cache size vs MHA")

### 5f. Compare Parameter Counts Across Models (Table)

In [ ]:
# Theoretical parameter count calculator for Transformer models
def transformer_param_count(vocab_size, d_model, num_layers, d_ff, num_heads,
                            max_seq_len=0, tied_embeddings=False):
    """Calculate total parameters for a Transformer.

    Components:
    1. Token embedding: vocab_size * d_model
    2. Position embedding: max_seq_len * d_model (if learned, 0 for RoPE)
    3. Per layer:
       - Attention: 4 * d_model^2 (Q, K, V, O projections)
       - FFN: 2 * d_model * d_ff (up + down, or 3 * d_model * d_ff for SwiGLU)
       - LayerNorm: 2 * d_model (or d_model for RMSNorm) x2 per layer
    4. Final LayerNorm: d_model
    5. LM head: vocab_size * d_model (often tied with embedding)
    """
    embed = vocab_size * d_model
    pos_embed = max_seq_len * d_model
    attn_per_layer = 4 * d_model * d_model  # Q, K, V, O
    ffn_per_layer = 2 * d_model * d_ff
    norm_per_layer = 2 * 2 * d_model  # 2 norms, each with weight + bias
    per_layer = attn_per_layer + ffn_per_layer + norm_per_layer
    total_layers = num_layers * per_layer
    final_norm = 2 * d_model
    lm_head = 0 if tied_embeddings else vocab_size * d_model
    total = embed + pos_embed + total_layers + final_norm + lm_head
    return total

configs = {
    "GPT-2 Small":  (50257, 768, 12, 3072, 12, 1024, True),
    "GPT-2 Medium": (50257, 1024, 24, 4096, 16, 1024, True),
    "GPT-2 Large":  (50257, 1280, 36, 5120, 20, 1024, True),
    "GPT-2 XL":     (50257, 1600, 48, 6400, 25, 1024, True),
    "BERT-base":    (30522, 768, 12, 3072, 12, 512, False),
    "BERT-large":   (30522, 1024, 24, 4096, 16, 512, False),
}

print(f"{'Model':<16} {'Calculated':>14} {'Official':>14}")
print("-" * 46)
official = {"GPT-2 Small": 124_000_000, "GPT-2 Medium": 355_000_000,
            "GPT-2 Large": 774_000_000, "GPT-2 XL": 1_500_000_000,
            "BERT-base": 110_000_000, "BERT-large": 340_000_000}
for name, (v, d, n, f, h, s, tied) in configs.items():
    calc = transformer_param_count(v, d, n, f, h, s, tied)
    off = official.get(name, 0)
    print(f"{name:<16} {calc:>14,} {off:>14,}")

---
# Section 6: Embeddings

Embeddings map text to dense vectors. They are the foundation of semantic search, RAG, and clustering.

**Key concepts:**
- **CLS pooling**: Use the [CLS] token embedding (BERT default).
- **Mean pooling**: Average all token embeddings (often better for similarity).
- **Anisotropy**: When embeddings cluster in a narrow cone, cosine similarity loses discrimination.

**Interview tip:** Know pooling strategies, why mean pooling often beats CLS, and how to check embedding quality.

### 6a. Get BERT Embeddings (CLS, Mean Pooling, Max Pooling)

In [ ]:
import torch
import numpy as np

try:
    from transformers import AutoTokenizer, AutoModel

    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModel.from_pretrained("bert-base-uncased")
    model.eval()

    def get_bert_embeddings(texts, pooling="mean"):
        """Get embeddings using different pooling strategies.

        Args:
            texts: list of strings
            pooling: 'cls', 'mean', or 'max'
        Returns:
            numpy array of shape (len(texts), hidden_size)
        """
        encoded = tokenizer(texts, padding=True, truncation=True,
                           max_length=128, return_tensors="pt")
        with torch.no_grad():
            outputs = model(**encoded)

        hidden_states = outputs.last_hidden_state  # (batch, seq, hidden)
        attention_mask = encoded['attention_mask']   # (batch, seq)

        if pooling == "cls":
            # Use [CLS] token (position 0)
            embeddings = hidden_states[:, 0, :]
        elif pooling == "mean":
            # Mean of all non-padding tokens
            mask_expanded = attention_mask.unsqueeze(-1).float()  # (batch, seq, 1)
            sum_embeddings = (hidden_states * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            embeddings = sum_embeddings / sum_mask
        elif pooling == "max":
            # Max pool across sequence dimension (ignoring padding)
            mask_expanded = attention_mask.unsqueeze(-1).float()
            hidden_states[mask_expanded == 0] = -1e9
            embeddings = hidden_states.max(dim=1).values
        else:
            raise ValueError(f"Unknown pooling: {pooling}")

        return embeddings.numpy()

    texts = [
        "The cat sat on the mat.",
        "A dog is lying on the rug.",
        "Python is a programming language.",
    ]

    for pool in ["cls", "mean", "max"]:
        embs = get_bert_embeddings(texts, pooling=pool)
        # Compute pairwise cosine similarity
        from sklearn.metrics.pairwise import cosine_similarity
        sim = cosine_similarity(embs)
        print(f"\n{pool.upper()} pooling - similarity matrix:")
        print(np.round(sim, 3))

except ImportError:
    print("Install transformers: pip install transformers")

### 6b. Get Sentence Embeddings with sentence-transformers

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    # all-MiniLM-L6-v2 is a small, fast, high-quality model
    model = SentenceTransformer('all-MiniLM-L6-v2')

    sentences = [
        "The weather is lovely today.",
        "It's so sunny outside!",
        "He drove to the store.",
        "The stock market crashed.",
    ]

    embeddings = model.encode(sentences)
    print(f"Embedding shape: {embeddings.shape}")  # (4, 384)

    sim = cosine_similarity(embeddings)
    print("\nSimilarity matrix:")
    for i, s in enumerate(sentences):
        print(f"  [{i}] {s}")
    print(np.round(sim, 3))

except ImportError:
    print("Install: pip install sentence-transformers")

### 6c. Compare Embedding Models on a Sample Query Set

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np
    import time

    query = "How to learn machine learning?"
    documents = [
        "Machine learning tutorials for beginners",
        "Best online courses for ML and AI",
        "Cooking recipes for Italian food",
        "The stock market performance in 2024",
        "Deep learning resources and books",
    ]

    model_names = ['all-MiniLM-L6-v2']  # keep small for demo
    # Add more: 'all-mpnet-base-v2', 'paraphrase-MiniLM-L6-v2'

    for mname in model_names:
        model = SentenceTransformer(mname)
        start = time.time()
        q_emb = model.encode([query])
        d_embs = model.encode(documents)
        elapsed = time.time() - start

        scores = cosine_similarity(q_emb, d_embs)[0]
        ranking = np.argsort(-scores)
        print(f"\nModel: {mname} (dim={d_embs.shape[1]}, time={elapsed:.2f}s)")
        for rank, idx in enumerate(ranking):
            print(f"  {rank+1}. [{scores[idx]:.3f}] {documents[idx]}")

except ImportError:
    print("Install: pip install sentence-transformers")

### 6d. Visualize Embeddings with t-SNE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Create synthetic embeddings for visualization
np.random.seed(42)
n_per_class = 30
dim = 64

# 3 clusters with some overlap
class_centers = [np.random.randn(dim) * 2 for _ in range(3)]
labels = []
embeddings = []
class_names = ["Sports", "Technology", "Cooking"]

for i, center in enumerate(class_centers):
    cluster = center + np.random.randn(n_per_class, dim) * 0.5
    embeddings.append(cluster)
    labels.extend([i] * n_per_class)

embeddings = np.vstack(embeddings)
labels = np.array(labels)

# t-SNE reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=20)
emb_2d = tsne.fit_transform(embeddings)

plt.figure(figsize=(8, 6))
for i, name in enumerate(class_names):
    mask = labels == i
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=name, alpha=0.7, s=50)
plt.legend()
plt.title("t-SNE Visualization of Embeddings")
plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
plt.tight_layout()
plt.savefig("tsne_embeddings.png", dpi=100)
plt.show()

### 6e. Check Anisotropy of an Embedding Space

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def check_anisotropy(embeddings, n_samples=500):
    """Check if embeddings are anisotropic (clustered in a narrow cone).

    Anisotropy = average pairwise cosine similarity of random pairs.
    High anisotropy (close to 1.0) = bad: all vectors point similarly.
    Low anisotropy (close to 0.0) = good: vectors well spread out.

    Modern embedding models should have low anisotropy.
    """
    n = min(n_samples, len(embeddings))
    indices = np.random.choice(len(embeddings), n, replace=False)
    sample = embeddings[indices]

    # Compute all pairwise similarities
    sim_matrix = cosine_similarity(sample)

    # Get upper triangle (exclude diagonal)
    upper = sim_matrix[np.triu_indices(n, k=1)]

    return {
        "mean_similarity": float(np.mean(upper)),
        "std_similarity": float(np.std(upper)),
        "min_similarity": float(np.min(upper)),
        "max_similarity": float(np.max(upper)),
    }

# Test with isotropic (good) vs anisotropic (bad) embeddings
np.random.seed(42)

# Good: random directions
good_embs = np.random.randn(200, 128)
good_embs = good_embs / np.linalg.norm(good_embs, axis=1, keepdims=True)

# Bad: all shifted toward same direction
bias = np.ones(128) * 5
bad_embs = np.random.randn(200, 128) + bias
bad_embs = bad_embs / np.linalg.norm(bad_embs, axis=1, keepdims=True)

print("Isotropic embeddings (good):")
print(f"  {check_anisotropy(good_embs)}")
print("\nAnisotropic embeddings (bad):")
print(f"  {check_anisotropy(bad_embs)}")

### 6f. Normalize Embeddings and Compare Before/After Similarity Distributions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

np.random.seed(42)
# Simulate embeddings with varying magnitudes
raw_embs = np.random.randn(100, 64) * np.random.uniform(0.5, 5.0, (100, 1))

# Normalize to unit length
norm_embs = raw_embs / np.linalg.norm(raw_embs, axis=1, keepdims=True)

# Compute pairwise similarities
raw_sim = cosine_similarity(raw_embs)
norm_sim = cosine_similarity(norm_embs)

# Extract upper triangles
raw_upper = raw_sim[np.triu_indices(100, k=1)]
norm_upper = norm_sim[np.triu_indices(100, k=1)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw_upper, bins=50, alpha=0.7, color='steelblue')
axes[0].set_title(f"Before Normalization\nmean={raw_upper.mean():.3f}")
axes[0].set_xlabel("Cosine Similarity")
axes[1].hist(norm_upper, bins=50, alpha=0.7, color='coral')
axes[1].set_title(f"After Normalization\nmean={norm_upper.mean():.3f}")
axes[1].set_xlabel("Cosine Similarity")
plt.tight_layout()
plt.savefig("normalization_effect.png", dpi=100)
plt.show()
print("Cosine similarity is magnitude-invariant, so distributions should be identical.")
print("For DOT PRODUCT similarity, normalization matters a lot!")

---
# Section 7: Semantic Search with FAISS

FAISS (Facebook AI Similarity Search) is the standard library for fast vector similarity search.

**Key index types:**
- **Flat**: Exact search, O(n) per query. Best recall, slowest.
- **IVF**: Inverted file index. Clusters vectors, searches nearby clusters. nprobe controls speed/recall tradeoff.
- **HNSW**: Hierarchical Navigable Small World graph. Fast, high recall, but more memory.
- **PQ**: Product Quantization. Compresses vectors, less memory, lower recall.
- **IVF-PQ**: Combines clustering with compression. Best for very large datasets.

**Interview tip:** Know the tradeoffs. Be able to explain nprobe, nlist, M (HNSW links), nbits (PQ).

### 7a. Build a Flat (Exact) FAISS Index

In [ ]:
try:
    import faiss
    import numpy as np

    # Create synthetic data: 10000 vectors of dimension 128
    np.random.seed(42)
    d = 128
    n = 10000
    db_vectors = np.random.randn(n, d).astype('float32')

    # Normalize for cosine similarity (FAISS IndexFlatIP with normalized = cosine)
    faiss.normalize_L2(db_vectors)

    # Build flat index (exact inner product search)
    index = faiss.IndexFlatIP(d)  # IP = Inner Product (= cosine for normalized)
    index.add(db_vectors)
    print(f"Index size: {index.ntotal} vectors")

    # Search
    query = np.random.randn(1, d).astype('float32')
    faiss.normalize_L2(query)
    k = 5
    scores, indices = index.search(query, k)
    print(f"Top-{k} results: indices={indices[0]}, scores={scores[0].round(4)}")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7b. Build an IVF Index with nprobe Tuning

In [ ]:
try:
    import faiss
    import numpy as np
    import time

    np.random.seed(42)
    d, n = 128, 100000
    db = np.random.randn(n, d).astype('float32')
    faiss.normalize_L2(db)

    # IVF: partition space into nlist clusters
    nlist = 100  # number of Voronoi cells
    quantizer = faiss.IndexFlatIP(d)  # coarse quantizer
    index = faiss.IndexIVFFlat(quantizer, d, nlist, faiss.METRIC_INNER_PRODUCT)

    # Must train before adding vectors
    index.train(db)
    index.add(db)
    print(f"IVF index: {index.ntotal} vectors, {nlist} clusters")

    # Compare nprobe values (how many clusters to search)
    query = np.random.randn(1, d).astype('float32')
    faiss.normalize_L2(query)

    # Get ground truth with flat search
    flat_index = faiss.IndexFlatIP(d)
    flat_index.add(db)
    _, gt = flat_index.search(query, 10)
    gt_set = set(gt[0])

    print(f"\n{'nprobe':<8} {'Time (ms)':<12} {'Recall@10':<12}")
    for nprobe in [1, 5, 10, 20, 50, 100]:
        index.nprobe = nprobe
        start = time.time()
        for _ in range(100):
            scores, ids = index.search(query, 10)
        elapsed = (time.time() - start) * 10  # ms per query
        recall = len(set(ids[0]) & gt_set) / len(gt_set)
        print(f"{nprobe:<8} {elapsed:<12.2f} {recall:<12.2f}")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7c. Build an HNSW Index

In [ ]:
try:
    import faiss
    import numpy as np
    import time

    np.random.seed(42)
    d, n = 128, 100000
    db = np.random.randn(n, d).astype('float32')
    faiss.normalize_L2(db)

    # HNSW: M = number of links per node (higher = better recall, more memory)
    M = 32
    index = faiss.IndexHNSWFlat(d, M)
    index.metric_type = faiss.METRIC_INNER_PRODUCT

    start = time.time()
    index.add(db)
    build_time = time.time() - start
    print(f"HNSW index built in {build_time:.2f}s, {index.ntotal} vectors, M={M}")

    # Search
    query = np.random.randn(5, d).astype('float32')
    faiss.normalize_L2(query)

    # efSearch controls search quality (higher = better recall, slower)
    index.hnsw.efSearch = 64
    start = time.time()
    scores, ids = index.search(query, 10)
    elapsed = (time.time() - start) * 1000
    print(f"Search time: {elapsed:.1f}ms for {len(query)} queries")
    print(f"Top result scores: {scores[0][:5].round(4)}")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7d. Build an IVF-PQ Index (Compressed)

In [ ]:
try:
    import faiss
    import numpy as np

    np.random.seed(42)
    d, n = 128, 100000
    db = np.random.randn(n, d).astype('float32')
    faiss.normalize_L2(db)

    # IVF-PQ: IVF for coarse search + PQ for compression
    nlist = 100     # Voronoi cells
    m = 16          # number of subquantizers (must divide d)
    nbits = 8       # bits per subquantizer (256 centroids per sub-space)

    index = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, nlist, m, nbits,
                             faiss.METRIC_INNER_PRODUCT)
    index.train(db)
    index.add(db)
    index.nprobe = 10

    # Memory comparison
    flat_mem = n * d * 4 / (1024**2)  # MB
    pq_mem = n * m * nbits / 8 / (1024**2)  # approximate MB
    print(f"Flat index memory: ~{flat_mem:.1f} MB")
    print(f"IVF-PQ index memory: ~{pq_mem:.1f} MB")
    print(f"Compression ratio: ~{flat_mem/pq_mem:.1f}x")

    # Search
    query = np.random.randn(1, d).astype('float32')
    faiss.normalize_L2(query)
    scores, ids = index.search(query, 5)
    print(f"\nTop-5 IVF-PQ results: {ids[0]}, scores: {scores[0].round(4)}")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7e. Search Comparison: Recall and Speed Across Index Types

In [ ]:
try:
    import faiss
    import numpy as np
    import time

    np.random.seed(42)
    d, n, nq = 128, 50000, 100
    db = np.random.randn(n, d).astype('float32')
    queries = np.random.randn(nq, d).astype('float32')
    faiss.normalize_L2(db)
    faiss.normalize_L2(queries)

    k = 10

    # Ground truth with flat
    flat = faiss.IndexFlatIP(d)
    flat.add(db)
    _, gt = flat.search(queries, k)

    def compute_recall(pred, gt, k):
        recalls = []
        for i in range(len(pred)):
            recalls.append(len(set(pred[i]) & set(gt[i])) / k)
        return np.mean(recalls)

    results = []

    # 1. Flat
    start = time.time()
    scores, ids = flat.search(queries, k)
    elapsed = (time.time() - start) * 1000 / nq
    results.append(("Flat (exact)", 1.0, elapsed))

    # 2. IVF
    nlist = 100
    ivf = faiss.IndexIVFFlat(faiss.IndexFlatIP(d), d, nlist, faiss.METRIC_INNER_PRODUCT)
    ivf.train(db); ivf.add(db); ivf.nprobe = 10
    start = time.time()
    _, ids = ivf.search(queries, k)
    elapsed = (time.time() - start) * 1000 / nq
    results.append(("IVF (nprobe=10)", compute_recall(ids, gt, k), elapsed))

    # 3. HNSW
    hnsw = faiss.IndexHNSWFlat(d, 32)
    hnsw.metric_type = faiss.METRIC_INNER_PRODUCT
    hnsw.add(db); hnsw.hnsw.efSearch = 64
    start = time.time()
    _, ids = hnsw.search(queries, k)
    elapsed = (time.time() - start) * 1000 / nq
    results.append(("HNSW (ef=64)", compute_recall(ids, gt, k), elapsed))

    print(f"{'Index':<22} {'Recall@10':<12} {'ms/query':<12}")
    print("-" * 46)
    for name, recall, ms in results:
        print(f"{name:<22} {recall:<12.4f} {ms:<12.3f}")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7f. Add/Remove Vectors from Index

In [ ]:
try:
    import faiss
    import numpy as np

    d = 64
    db = np.random.randn(100, d).astype('float32')

    # Use IndexIDMap to support remove by ID
    index = faiss.IndexIDMap(faiss.IndexFlatL2(d))

    # Add with explicit IDs
    ids = np.arange(100).astype('int64')
    index.add_with_ids(db, ids)
    print(f"After adding: {index.ntotal} vectors")

    # Remove specific IDs
    remove_ids = np.array([10, 20, 30], dtype='int64')
    index.remove_ids(faiss.IDSelectorBatch(remove_ids))
    print(f"After removing 3: {index.ntotal} vectors")

    # Add new vectors
    new_vectors = np.random.randn(5, d).astype('float32')
    new_ids = np.array([200, 201, 202, 203, 204], dtype='int64')
    index.add_with_ids(new_vectors, new_ids)
    print(f"After adding 5 new: {index.ntotal} vectors")

except ImportError:
    print("Install FAISS: pip install faiss-cpu")

### 7g. Save and Load Index

In [ ]:
try:
    import faiss
    import numpy as np
    import os

    d = 64
    db = np.random.randn(1000, d).astype('float32')
    index = faiss.IndexFlatL2(d)
    index.add(db)

    # Save
    faiss.write_index(index, "my_index.faiss")
    file_size = os.path.getsize("my_index.faiss") / 1024
    print(f"Saved index: {file_size:.1f} KB")

    # Load
    loaded_index = faiss.read_index("my_index.faiss")
    print(f"Loaded index: {loaded_index.ntotal} vectors")

    # Verify same results
    query = np.random.randn(1, d).astype('float32')
    d1, i1 = index.search(query, 5)
    d2, i2 = loaded_index.search(query, 5)
    print(f"Results match: {np.array_equal(i1, i2)}")

    # Cleanup
    os.remove("my_index.faiss")
except ImportError:
    print("Install FAISS: pip install faiss-cpu")

---
# Section 8: BM25 Search

BM25 is the classic lexical search algorithm. It's a bag-of-words model that scores documents by term frequency, inverse document frequency, and document length.

**Formula:** score(q, d) = sum over terms t in q: IDF(t) * (tf(t,d) * (k1 + 1)) / (tf(t,d) + k1 * (1 - b + b * |d|/avgdl))

**Interview tip:** Know BM25 formula, parameters k1 and b, and why hybrid (BM25 + dense) often beats either alone.

### 8a. BM25 from Scratch (Full Class)

In [ ]:
import numpy as np
import math
from collections import Counter

class BM25:
    """BM25 implementation from scratch.

    Parameters:
        k1: term frequency saturation. Higher = more weight to repeated terms. Default 1.5.
        b: document length normalization. 0 = no normalization, 1 = full. Default 0.75.
    """
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b

    def fit(self, corpus):
        """Index a corpus (list of strings)."""
        self.corpus = corpus
        self.N = len(corpus)

        # Tokenize (simple whitespace + lowercase)
        self.doc_tokens = [doc.lower().split() for doc in corpus]
        self.doc_lens = [len(doc) for doc in self.doc_tokens]
        self.avgdl = np.mean(self.doc_lens)

        # Term frequencies per document
        self.tf = [Counter(doc) for doc in self.doc_tokens]

        # Document frequency: how many docs contain each term
        self.df = Counter()
        for doc in self.doc_tokens:
            for term in set(doc):
                self.df[term] += 1

        # Precompute IDF
        self.idf = {}
        for term, df in self.df.items():
            # Standard BM25 IDF formula
            self.idf[term] = math.log((self.N - df + 0.5) / (df + 0.5) + 1)

    def score(self, query):
        """Score all documents against a query string."""
        query_terms = query.lower().split()
        scores = np.zeros(self.N)

        for term in query_terms:
            if term not in self.idf:
                continue
            idf = self.idf[term]
            for i in range(self.N):
                tf = self.tf[i].get(term, 0)
                dl = self.doc_lens[i]
                # BM25 scoring formula
                numerator = tf * (self.k1 + 1)
                denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[i] += idf * numerator / denominator
        return scores

    def search(self, query, top_k=5):
        """Return top-k documents with scores."""
        scores = self.score(query)
        top_indices = np.argsort(-scores)[:top_k]
        return [(int(i), float(scores[i]), self.corpus[i]) for i in top_indices]

# Test
corpus = [
    "the cat sat on the mat",
    "the dog played in the park",
    "a cat and a dog became friends",
    "machine learning is a subset of artificial intelligence",
    "deep learning uses neural networks with many layers",
    "the cat chased the dog around the yard",
]

bm25 = BM25()
bm25.fit(corpus)

query = "cat dog friends"
results = bm25.search(query, top_k=3)
print(f"Query: '{query}'")
for rank, (idx, score, doc) in enumerate(results):
    print(f"  {rank+1}. [{score:.4f}] {doc}")

### 8b. BM25 Using rank_bm25 Library

In [ ]:
try:
    from rank_bm25 import BM25Okapi

    corpus = [
        "the cat sat on the mat",
        "the dog played in the park",
        "a cat and a dog became friends",
        "machine learning is a subset of artificial intelligence",
        "deep learning uses neural networks with many layers",
    ]

    tokenized_corpus = [doc.lower().split() for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)

    query = "machine learning neural"
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)

    print(f"Query: '{query}'")
    for i in np.argsort(-scores):
        print(f"  [{scores[i]:.4f}] {corpus[i]}")

except ImportError:
    print("Install: pip install rank_bm25")

### 8c. Hybrid Search: BM25 + Dense with Reciprocal Rank Fusion (RRF)

In [ ]:
import numpy as np

def reciprocal_rank_fusion(rankings, k=60):
    """Combine multiple ranked lists using RRF.

    RRF score for document d = sum over rankings r: 1 / (k + rank_r(d))

    k=60 is the standard value from the original paper.
    RRF is robust and doesn't require score normalization.
    """
    doc_scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            if doc_id not in doc_scores:
                doc_scores[doc_id] = 0
            doc_scores[doc_id] += 1.0 / (k + rank + 1)  # rank is 0-indexed

    # Sort by RRF score descending
    sorted_docs = sorted(doc_scores.items(), key=lambda x: -x[1])
    return sorted_docs

# Simulate BM25 ranking and dense ranking
bm25_ranking = [2, 0, 5, 1, 3, 4]    # BM25 ranks doc 2 first
dense_ranking = [5, 2, 1, 0, 4, 3]    # Dense ranks doc 5 first

# Documents
docs = {
    0: "the cat sat on the mat",
    1: "the dog played in the park",
    2: "a cat and a dog became friends",
    3: "machine learning basics",
    4: "deep learning with neural networks",
    5: "cats and dogs are popular pets",
}

fused = reciprocal_rank_fusion([bm25_ranking, dense_ranking])
print("Hybrid ranking (RRF):")
for doc_id, score in fused:
    print(f"  [{score:.4f}] Doc {doc_id}: {docs[doc_id]}")

print("\nNote: Doc 2 appears near top of both, so it ranks highest in fusion")

### 8d. Compare BM25 vs Dense vs Hybrid on Sample Queries

In [ ]:
import numpy as np

# Simulated scenario showing where each method excels
corpus = [
    "Python programming language tutorial for beginners",      # 0
    "Java enterprise application development guide",           # 1
    "How to cook Italian pasta from scratch",                  # 2
    "Python snake species found in tropical forests",          # 3
    "Introduction to machine learning with Python",            # 4
    "Software engineering best practices and design patterns", # 5
]

# Simulated rankings for query: "Python programming"
# BM25 excels at exact keyword matching
bm25_ranking = [0, 3, 4, 1, 5, 2]  # BM25 also ranks "Python snake" high (keyword match)
# Dense excels at semantic understanding
dense_ranking = [0, 4, 1, 5, 2, 3]  # Dense correctly ranks "Python snake" low

hybrid = reciprocal_rank_fusion([bm25_ranking, dense_ranking])

print("Query: 'Python programming'")
print(f"{'Method':<10} {'Ranking':>40}")
print(f"{'BM25':<10} {str([corpus[i][:40] for i in bm25_ranking[:3]]):>40}")
print(f"{'Dense':<10} {str([corpus[i][:40] for i in dense_ranking[:3]]):>40}")
print(f"{'Hybrid':<10} {str([corpus[r[0]][:40] for r in hybrid[:3]]):>40}")

print("\nKey takeaways:")
print("  - BM25: good for exact matches, bad for semantic understanding")
print("  - Dense: good for semantics, may miss exact keyword matches")
print("  - Hybrid: combines strengths of both")

---
# Section 9: Cross-Encoder Reranking

Cross-encoders jointly encode (query, document) pairs and produce a relevance score. They are much more accurate than bi-encoders but too slow for initial retrieval.

**Pipeline:** Bi-encoder retrieves top-k candidates -> Cross-encoder reranks them.

**Interview tip:** Know the difference between bi-encoder (independent encoding, fast) and cross-encoder (joint encoding, accurate). Know why we need a two-stage pipeline.

### 9a-9b. Load Cross-Encoder and Score Query-Document Pairs

In [ ]:
try:
    from sentence_transformers import CrossEncoder

    # Small cross-encoder model
    model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    query = "What is machine learning?"
    documents = [
        "Machine learning is a subset of AI that learns from data.",
        "The weather forecast predicts rain tomorrow.",
        "Deep learning is a type of machine learning using neural networks.",
        "Python is a popular programming language.",
        "ML algorithms can be supervised or unsupervised.",
    ]

    # Score each (query, document) pair
    pairs = [(query, doc) for doc in documents]
    scores = model.predict(pairs)

    print(f"Query: '{query}'\n")
    for score, doc in sorted(zip(scores, documents), reverse=True):
        print(f"  [{score:.4f}] {doc}")

except ImportError:
    print("Install: pip install sentence-transformers")

### 9c. Rerank a List of Candidates

In [ ]:
def rerank(query, candidates, cross_encoder, top_k=None):
    """Rerank candidates using a cross-encoder.

    Args:
        query: search query string
        candidates: list of (doc_id, text) tuples from initial retrieval
        cross_encoder: CrossEncoder model
        top_k: return top-k results (None = return all)
    Returns:
        list of (doc_id, score, text) sorted by relevance
    """
    pairs = [(query, text) for _, text in candidates]
    scores = cross_encoder.predict(pairs)

    # Sort by score descending
    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    results = [(doc_id, float(score), text) for (doc_id, text), score in ranked]

    if top_k:
        results = results[:top_k]
    return results

try:
    from sentence_transformers import CrossEncoder
    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    query = "best programming language for data science"
    # Simulated initial retrieval results (from bi-encoder)
    candidates = [
        (42, "Python is widely used in data science and machine learning."),
        (17, "JavaScript is the language of the web."),
        (88, "R is a statistical programming language popular with data scientists."),
        (31, "Data science combines statistics, programming, and domain expertise."),
        (55, "C++ is fast and used in game development."),
    ]

    reranked = rerank(query, candidates, ce, top_k=3)
    print(f"Query: '{query}'\n")
    print("After reranking:")
    for doc_id, score, text in reranked:
        print(f"  [{score:.4f}] Doc {doc_id}: {text}")

except ImportError:
    print("Install: pip install sentence-transformers")

### 9d. Compare Retrieval Quality Before and After Reranking (MRR, NDCG)

In [ ]:
import numpy as np

def mrr(ranked_relevant):
    """Mean Reciprocal Rank: 1/rank of first relevant result."""
    for i, rel in enumerate(ranked_relevant):
        if rel:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(ranked_relevant, k):
    """NDCG@K: normalized discounted cumulative gain."""
    dcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ranked_relevant[:k]))
    ideal = sorted(ranked_relevant, reverse=True)[:k]
    idcg = sum(rel / np.log2(i + 2) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

# Simulate: relevance labels for each document
# 1 = relevant, 0 = not relevant
# Before reranking (bi-encoder order)
before_relevance = [0, 1, 0, 0, 1]  # relevant docs at positions 2, 5
# After reranking (cross-encoder order)
after_relevance = [1, 1, 0, 0, 0]   # relevant docs moved to positions 1, 2

print("Before reranking:")
print(f"  Relevance order: {before_relevance}")
print(f"  MRR: {mrr(before_relevance):.4f}")
print(f"  NDCG@5: {ndcg_at_k(before_relevance, 5):.4f}")

print("\nAfter reranking:")
print(f"  Relevance order: {after_relevance}")
print(f"  MRR: {mrr(after_relevance):.4f}")
print(f"  NDCG@5: {ndcg_at_k(after_relevance, 5):.4f}")

### 9e. Batch Reranking for Efficiency

In [ ]:
def batch_rerank(query, candidates, cross_encoder, batch_size=32, top_k=None):
    """Rerank with batching for efficiency with large candidate sets."""
    pairs = [(query, text) for _, text in candidates]

    # Score in batches
    all_scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        scores = cross_encoder.predict(batch)
        all_scores.extend(scores)

    ranked = sorted(zip(candidates, all_scores), key=lambda x: -x[1])
    results = [(doc_id, float(score), text) for (doc_id, text), score in ranked]
    return results[:top_k] if top_k else results

try:
    from sentence_transformers import CrossEncoder
    import time

    ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    # Simulate large candidate set
    candidates = [(i, f"Document number {i} about topic {i % 10}") for i in range(100)]
    query = "document about topic 5"

    start = time.time()
    results = batch_rerank(query, candidates, ce, batch_size=32, top_k=5)
    elapsed = time.time() - start

    print(f"Reranked {len(candidates)} candidates in {elapsed:.2f}s")
    print("Top 5:")
    for doc_id, score, text in results:
        print(f"  [{score:.4f}] {text}")

except ImportError:
    print("Install: pip install sentence-transformers")

---
# Section 10: Evaluation Metrics

Standard metrics for evaluating retrieval and ranking systems.

**Interview tip:** Know NDCG (graded relevance), MRR (first relevant), MAP (binary relevance), Recall@K. Be ready to implement any from scratch.

### 10a. NDCG@K from Scratch

In [ ]:
import numpy as np

def dcg_at_k(relevances, k):
    """Discounted Cumulative Gain at K.
    Higher ranks get more weight via log discount.
    """
    relevances = np.array(relevances[:k])
    positions = np.arange(1, len(relevances) + 1)
    discounts = np.log2(positions + 1)
    return np.sum(relevances / discounts)

def ndcg_at_k(relevances, k):
    """Normalized DCG: DCG / ideal DCG.
    Range [0, 1]. 1.0 = perfect ranking.
    """
    dcg = dcg_at_k(relevances, k)
    ideal_relevances = sorted(relevances, reverse=True)
    idcg = dcg_at_k(ideal_relevances, k)
    return dcg / idcg if idcg > 0 else 0.0

# Example: graded relevance (3=highly relevant, 2=relevant, 1=somewhat, 0=irrelevant)
relevances = [3, 0, 2, 1, 0, 0, 1]
for k in [1, 3, 5, 7]:
    print(f"NDCG@{k} = {ndcg_at_k(relevances, k):.4f}")

perfect = sorted(relevances, reverse=True)
print(f"\nPerfect ranking: {perfect}")
print(f"NDCG@7 of perfect = {ndcg_at_k(perfect, 7):.4f}")

### 10b. MRR from Scratch

In [ ]:
import numpy as np

def mrr(ranked_results):
    """Mean Reciprocal Rank for a single query.
    ranked_results: list of 0/1 relevance labels in ranked order.
    MRR = 1 / (position of first relevant result).
    """
    for i, rel in enumerate(ranked_results):
        if rel:
            return 1.0 / (i + 1)
    return 0.0

def mean_mrr(all_rankings):
    """Average MRR across multiple queries."""
    return np.mean([mrr(r) for r in all_rankings])

queries_results = [
    [0, 0, 1, 0, 1],  # first relevant at position 3 -> RR = 1/3
    [1, 0, 0, 0, 0],  # first relevant at position 1 -> RR = 1
    [0, 1, 0, 1, 0],  # first relevant at position 2 -> RR = 1/2
]

for i, r in enumerate(queries_results):
    print(f"Query {i}: {r} -> RR = {mrr(r):.4f}")
print(f"\nMean MRR = {mean_mrr(queries_results):.4f}")

### 10c. MAP from Scratch

In [ ]:
import numpy as np

def average_precision(relevances):
    """Average Precision for a single query.
    AP = average of precision@k for each k where result is relevant.
    """
    relevant_count = 0
    precision_sum = 0.0
    for i, rel in enumerate(relevances):
        if rel:
            relevant_count += 1
            precision_at_k = relevant_count / (i + 1)
            precision_sum += precision_at_k
    total_relevant = sum(relevances)
    return precision_sum / total_relevant if total_relevant > 0 else 0.0

def mean_average_precision(all_relevances):
    return np.mean([average_precision(r) for r in all_relevances])

rankings = [
    [1, 0, 1, 0, 1],  # AP = (1/1 + 2/3 + 3/5) / 3
    [0, 0, 1, 1, 0],  # AP = (1/3 + 2/4) / 2
]

for i, r in enumerate(rankings):
    print(f"Query {i}: {r} -> AP = {average_precision(r):.4f}")
print(f"\nMAP = {mean_average_precision(rankings):.4f}")

### 10d. Recall@K from Scratch

In [ ]:
def recall_at_k(retrieved, relevant, k):
    """Recall@K: fraction of relevant documents found in top-K."""
    retrieved_at_k = set(retrieved[:k])
    return len(retrieved_at_k & relevant) / len(relevant) if relevant else 0.0

retrieved = [10, 5, 3, 7, 1, 8, 2, 6]
relevant = {3, 7, 8, 15}

for k in [1, 3, 5, 8]:
    print(f"Recall@{k} = {recall_at_k(retrieved, relevant, k):.4f}")

### 10e. Precision@K from Scratch

In [ ]:
def precision_at_k(retrieved, relevant, k):
    """Precision@K: fraction of top-K results that are relevant."""
    retrieved_at_k = set(retrieved[:k])
    return len(retrieved_at_k & relevant) / k if k > 0 else 0.0

retrieved = [10, 5, 3, 7, 1, 8, 2, 6]
relevant = {3, 7, 8, 15}

for k in [1, 3, 5, 8]:
    r = recall_at_k(retrieved, relevant, k)
    p = precision_at_k(retrieved, relevant, k)
    print(f"K={k}: Precision={p:.4f}, Recall={r:.4f}")

### 10f. F1 Score

In [ ]:
def f1_score(precision, recall):
    """Harmonic mean of precision and recall."""
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

for k in [1, 3, 5, 8]:
    p = precision_at_k(retrieved, relevant, k)
    r = recall_at_k(retrieved, relevant, k)
    f1 = f1_score(p, r)
    print(f"K={k}: P={p:.3f}, R={r:.3f}, F1={f1:.3f}")

### 10g. All Metrics Applied to a Sample Ranked Result Set

In [ ]:
import numpy as np

print("=" * 60)
print("COMPLETE EVALUATION EXAMPLE")
print("=" * 60)

eval_data = [
    {"query": "machine learning basics",
     "graded_rel": [3, 0, 2, 1, 0],
     "binary_rel": [1, 0, 1, 1, 0],
     "retrieved_ids": [1, 5, 3, 7, 9],
     "all_relevant_ids": {1, 3, 7, 11}},
    {"query": "python tutorial",
     "graded_rel": [0, 2, 3, 0, 1],
     "binary_rel": [0, 1, 1, 0, 1],
     "retrieved_ids": [4, 2, 8, 6, 10],
     "all_relevant_ids": {2, 8, 10}},
    {"query": "deep learning frameworks",
     "graded_rel": [2, 3, 0, 0, 1],
     "binary_rel": [1, 1, 0, 0, 1],
     "retrieved_ids": [12, 15, 3, 7, 20],
     "all_relevant_ids": {12, 15, 20, 25}},
]

for data in eval_data:
    q = data["query"]
    print(f"\nQuery: '{q}'")
    print(f"  NDCG@5:    {ndcg_at_k(data['graded_rel'], 5):.4f}")
    print(f"  MRR:       {mrr(data['binary_rel']):.4f}")
    print(f"  AP:        {average_precision(data['binary_rel']):.4f}")
    print(f"  P@3:       {precision_at_k(data['retrieved_ids'], data['all_relevant_ids'], 3):.4f}")
    print(f"  R@5:       {recall_at_k(data['retrieved_ids'], data['all_relevant_ids'], 5):.4f}")

print(f"\n--- Aggregate ---")
print(f"Mean NDCG@5: {np.mean([ndcg_at_k(d['graded_rel'], 5) for d in eval_data]):.4f}")
print(f"Mean MRR:    {mean_mrr([d['binary_rel'] for d in eval_data]):.4f}")
print(f"MAP:         {mean_average_precision([d['binary_rel'] for d in eval_data]):.4f}")

### 10h. BERTScore for Text Generation Evaluation

In [ ]:
# BERTScore measures similarity between generated and reference text
# using contextual BERT embeddings (token-level matching).

try:
    from bert_score import score as bert_score

    candidates = ["The cat sat on the mat.", "Machine learning is a field of study."]
    references = ["A cat was sitting on the mat.", "ML is an area of computer science."]

    P, R, F1 = bert_score(candidates, references, lang="en", verbose=False)
    for i in range(len(candidates)):
        print(f"Candidate: '{candidates[i]}'")
        print(f"Reference: '{references[i]}'")
        print(f"  Precision={P[i]:.4f}, Recall={R[i]:.4f}, F1={F1[i]:.4f}\n")
except ImportError:
    print("Install: pip install bert-score")
    print("BERTScore: token-level matching using BERT embeddings.")
    print("P = avg max-sim for candidate tokens, R = avg max-sim for reference tokens")

---
# Section 11: Loss Functions in PyTorch

Loss functions for training LLMs, embedding models, and rerankers.

**Interview tip:** Know cross-entropy for classification/LM, contrastive/triplet for embeddings, InfoNCE for CLIP-style, DPO for alignment.

### 11a. Cross-Entropy Loss (Manual + Torch)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def cross_entropy_manual(logits, target):
    """Cross-entropy from scratch. CE = -log(softmax(logits)[target])"""
    max_logits = np.max(logits, axis=-1, keepdims=True)
    log_sum_exp = np.log(np.sum(np.exp(logits - max_logits), axis=-1)) + max_logits.squeeze()
    target_logits = logits[np.arange(len(target)), target]
    loss = -target_logits + log_sum_exp
    return np.mean(loss)

np.random.seed(42)
logits_np = np.random.randn(4, 10)
targets_np = np.array([3, 7, 1, 5])

manual_loss = cross_entropy_manual(logits_np, targets_np)
torch_loss = F.cross_entropy(torch.tensor(logits_np, dtype=torch.float32),
                              torch.tensor(targets_np, dtype=torch.long))
print(f"Manual CE: {manual_loss:.4f}")
print(f"PyTorch CE: {torch_loss.item():.4f}")

### 11b. Binary Cross-Entropy with Logits

In [ ]:
import torch
import torch.nn.functional as F

def bce_with_logits_manual(logits, targets):
    """BCE with logits: max(x,0) - x*y + log(1+exp(-|x|))"""
    return torch.mean(
        torch.clamp(logits, min=0) - logits * targets
        + torch.log(1 + torch.exp(-torch.abs(logits)))
    )

logits = torch.randn(4, 5)
targets = torch.randint(0, 2, (4, 5)).float()
print(f"Manual BCE: {bce_with_logits_manual(logits, targets).item():.4f}")
print(f"PyTorch BCE: {F.binary_cross_entropy_with_logits(logits, targets).item():.4f}")

### 11c. Contrastive Loss (Siamese)

In [ ]:
import torch

def contrastive_loss(emb1, emb2, label, margin=1.0):
    """label=1: similar, label=0: dissimilar.
    Loss = y*d^2 + (1-y)*max(0, margin-d)^2"""
    dist = torch.pairwise_distance(emb1, emb2)
    loss = label * dist.pow(2) + (1 - label) * torch.clamp(margin - dist, min=0).pow(2)
    return loss.mean()

emb1 = torch.randn(4, 64)
emb2 = torch.randn(4, 64)
labels = torch.tensor([1, 1, 0, 0], dtype=torch.float)
print(f"Contrastive loss: {contrastive_loss(emb1, emb2, labels).item():.4f}")

### 11d. Triplet Loss with Semi-Hard Negative Mining

In [ ]:
import torch
import torch.nn.functional as F

def triplet_loss(anchor, positive, negative, margin=0.3):
    """Loss = max(0, d(a,p) - d(a,n) + margin)"""
    d_pos = F.pairwise_distance(anchor, positive)
    d_neg = F.pairwise_distance(anchor, negative)
    return torch.clamp(d_pos - d_neg + margin, min=0).mean()

def mine_semi_hard_negatives(anchors, positives, all_embs, margin=0.3):
    """Semi-hard: d(a,p) < d(a,n) < d(a,p) + margin"""
    d_pos = F.pairwise_distance(anchors, positives)
    negatives = []
    for i in range(len(anchors)):
        dists = F.pairwise_distance(anchors[i].unsqueeze(0), all_embs)
        mask = (dists > d_pos[i]) & (dists < d_pos[i] + margin)
        if mask.any():
            valid_dists = dists.clone(); valid_dists[~mask] = float('inf')
            neg_idx = valid_dists.argmin()
        else:
            neg_idx = dists.argmin()
        negatives.append(all_embs[neg_idx])
    return torch.stack(negatives)

anchors = torch.randn(8, 64)
positives = anchors + torch.randn(8, 64) * 0.1
all_embs = torch.randn(100, 64)
semi_hard = mine_semi_hard_negatives(anchors, positives, all_embs)
print(f"Triplet loss (semi-hard): {triplet_loss(anchors, positives, semi_hard).item():.4f}")

### 11e. InfoNCE / NT-Xent Loss with Temperature

In [ ]:
import torch
import torch.nn.functional as F

def info_nce_loss(query, positive_key, negative_keys, temperature=0.07):
    """InfoNCE (CLIP, SimCLR). Lower temp = sharper distribution."""
    query = F.normalize(query, dim=-1)
    positive_key = F.normalize(positive_key, dim=-1)
    negative_keys = F.normalize(negative_keys, dim=-1)

    pos_sim = torch.sum(query * positive_key, dim=-1, keepdim=True) / temperature
    neg_sim = query @ negative_keys.T / temperature
    logits = torch.cat([pos_sim, neg_sim], dim=-1)
    labels = torch.zeros(len(query), dtype=torch.long)
    return F.cross_entropy(logits, labels)

batch, dim, num_neg = 16, 128, 64
q = torch.randn(batch, dim)
p = torch.randn(batch, dim)
n = torch.randn(num_neg, dim)

for temp in [0.01, 0.07, 0.5, 1.0]:
    print(f"temp={temp}: loss={info_nce_loss(q, p, n, temp).item():.4f}")

### 11f. Multiple Negatives Ranking Loss (MNRL)

In [ ]:
import torch
import torch.nn.functional as F

def mnrl_loss(query_embs, positive_embs, temperature=0.05):
    """MNRL: in-batch negatives. Each positive is negative for other queries."""
    query_embs = F.normalize(query_embs, dim=-1)
    positive_embs = F.normalize(positive_embs, dim=-1)
    sim_matrix = query_embs @ positive_embs.T / temperature
    labels = torch.arange(len(query_embs))
    return F.cross_entropy(sim_matrix, labels)

for b in [8, 16, 32, 64, 128]:
    loss = mnrl_loss(torch.randn(b, 128), torch.randn(b, 128))
    print(f"batch={b:>4}: MNRL loss={loss.item():.4f}")

### 11g. Focal Loss

In [ ]:
import torch
import torch.nn.functional as F

def focal_loss(logits, targets, gamma=2.0):
    """Down-weights easy examples. gamma=0 -> standard CE."""
    ce = F.cross_entropy(logits, targets, reduction='none')
    pt = torch.exp(-ce)
    return ((1 - pt) ** gamma * ce).mean()

logits = torch.randn(10, 5)
targets = torch.randint(0, 5, (10,))
for g in [0.0, 1.0, 2.0, 5.0]:
    print(f"gamma={g}: {focal_loss(logits, targets, g).item():.4f}")
print(f"Standard CE: {F.cross_entropy(logits, targets).item():.4f}")

### 11h. Knowledge Distillation Loss

In [ ]:
import torch
import torch.nn.functional as F

def distillation_loss(student_logits, teacher_logits, true_labels, temperature=4.0, alpha=0.5):
    """KD loss = alpha * KL(soft_student || soft_teacher) * T^2 + (1-alpha) * CE(student, labels)"""
    soft_teacher = F.softmax(teacher_logits / temperature, dim=-1)
    soft_student = F.log_softmax(student_logits / temperature, dim=-1)
    soft_loss = F.kl_div(soft_student, soft_teacher, reduction='batchmean') * temperature ** 2
    hard_loss = F.cross_entropy(student_logits, true_labels)
    return alpha * soft_loss + (1 - alpha) * hard_loss

student = torch.randn(8, 10)
teacher = torch.randn(8, 10) * 2
labels = torch.randint(0, 10, (8,))
for temp in [1.0, 4.0, 8.0]:
    print(f"T={temp}: {distillation_loss(student, teacher, labels, temp).item():.4f}")

### 11i. DPO Loss (Direct Preference Optimization)

In [ ]:
import torch
import torch.nn.functional as F

def dpo_loss(policy_chosen_logps, policy_rejected_logps,
             ref_chosen_logps, ref_rejected_logps, beta=0.1):
    """DPO: aligns model with preferences without reward model.
    Loss = -log(sigmoid(beta * (log_ratio_chosen - log_ratio_rejected)))"""
    chosen_ratios = policy_chosen_logps - ref_chosen_logps
    rejected_ratios = policy_rejected_logps - ref_rejected_logps
    logits = beta * (chosen_ratios - rejected_ratios)
    loss = -F.logsigmoid(logits).mean()
    margin = (beta * (chosen_ratios - rejected_ratios)).mean().item()
    return loss, margin

batch = 16
pc = torch.randn(batch) * 0.5 - 2.0
pr = torch.randn(batch) * 0.5 - 3.0
rc = torch.randn(batch) * 0.5 - 2.5
rr = torch.randn(batch) * 0.5 - 2.5

for beta in [0.05, 0.1, 0.5]:
    loss, margin = dpo_loss(pc, pr, rc, rr, beta)
    print(f"beta={beta}: loss={loss.item():.4f}, margin={margin:.4f}")

### 11j. Label Smoothing Cross-Entropy

In [ ]:
import torch
import torch.nn.functional as F

def label_smoothing_ce(logits, targets, smoothing=0.1):
    """Smoothed targets: (1-eps)*one_hot + eps/C. Prevents overconfidence."""
    C = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    with torch.no_grad():
        smooth = torch.full_like(log_probs, smoothing / C)
        smooth.scatter_(1, targets.unsqueeze(1), 1.0 - smoothing + smoothing / C)
    return -(smooth * log_probs).sum(dim=-1).mean()

logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))
for eps in [0.0, 0.05, 0.1, 0.2]:
    print(f"smoothing={eps}: {label_smoothing_ce(logits, targets, eps).item():.4f}")
print(f"Standard CE: {F.cross_entropy(logits, targets).item():.4f}")

### 11k. Margin MSE Loss (for Reranker Distillation)

In [ ]:
import torch

def margin_mse_loss(s_pos, s_neg, t_pos, t_neg):
    """Match margin (score difference) between teacher and student."""
    return torch.mean((s_pos - s_neg - (t_pos - t_neg)) ** 2)

t_pos = torch.randn(16) + 2.0
t_neg = torch.randn(16)
s_pos = torch.randn(16, requires_grad=True)
s_neg = torch.randn(16, requires_grad=True)
loss = margin_mse_loss(s_pos, s_neg, t_pos, t_neg)
loss.backward()
print(f"Margin MSE loss: {loss.item():.4f}")

---
# Section 12: LoRA Implementation

LoRA adds small trainable low-rank matrices to frozen weights: W' = W + BA.

**Interview tip:** Know rank (r), alpha, why B is initialized to zero, and how to merge for inference.

### 12a. LoRA Linear Layer from Scratch

In [ ]:
import torch
import torch.nn as nn
import math

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, alpha=16):
        super().__init__()
        self.scaling = alpha / r
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)  # B=0 so LoRA starts as identity

    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

layer = LoRALinear(768, 768, r=8, alpha=16)
x = torch.randn(2, 10, 768)
print(f"Output: {layer(x).shape}")
total = sum(p.numel() for p in layer.parameters())
trainable = sum(p.numel() for p in layer.parameters() if p.requires_grad)
print(f"Total: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.2f}%)")

### 12b. Apply LoRA to a Pretrained Model

In [ ]:
import torch
import torch.nn as nn
import math

try:
    from transformers import AutoModel

    model = AutoModel.from_pretrained("bert-base-uncased")
    for param in model.parameters():
        param.requires_grad = False

    # Count LoRA params we would add to Q and V projections
    lora_params = 0
    r = 8
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(t in name for t in ["query", "value"]):
            lora_params += r * module.in_features + module.out_features * r

    total = sum(p.numel() for p in model.parameters())
    print(f"Base model: {total:,} params")
    print(f"LoRA params (r={r}, Q+V): {lora_params:,} ({100*lora_params/total:.2f}%)")
except ImportError:
    print("Install: pip install transformers")

### 12c. Merge LoRA Weights for Inference

In [ ]:
import torch
import torch.nn as nn
import math

class LoRAMergeable(nn.Module):
    def __init__(self, in_f, out_f, r=8, alpha=16):
        super().__init__()
        self.scaling = alpha / r
        self.linear = nn.Linear(in_f, out_f)
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.merged = False

    def merge(self):
        if not self.merged:
            self.linear.weight.data += self.scaling * (self.lora_B @ self.lora_A)
            self.merged = True

    def forward(self, x):
        if self.merged:
            return self.linear(x)
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

layer = LoRAMergeable(256, 256, r=8)
x = torch.randn(1, 10, 256)
out_before = layer(x)
layer.merge()
out_after = layer(x)
print(f"Max diff after merge: {(out_before - out_after).abs().max().item():.10f}")
print("After merge, no extra computation at inference!")

### 12d. Count Trainable vs Frozen Parameters

In [ ]:
import torch.nn as nn

def param_summary(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Total: {total:,}, Trainable: {trainable:,} ({100*trainable/total:.1f}%), Frozen: {total-trainable:,}")

# Example: freeze backbone, train head
model = nn.Sequential(nn.Linear(768, 3072), nn.ReLU(), nn.Linear(3072, 768), nn.Linear(768, 10))
for p in list(model.parameters())[:4]:  # freeze first two linears
    p.requires_grad = False
param_summary(model)

### 12e. Using HuggingFace PEFT Library for LoRA

In [ ]:
try:
    from transformers import AutoModelForSequenceClassification
    from peft import get_peft_model, LoraConfig, TaskType

    model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16,
                        lora_dropout=0.1, target_modules=["query", "value"])
    peft_model = get_peft_model(model, config)
    peft_model.print_trainable_parameters()
except ImportError:
    print("Install: pip install peft transformers")

### 12f. Save and Load LoRA Adapters

In [ ]:
try:
    from transformers import AutoModelForSequenceClassification
    from peft import get_peft_model, LoraConfig, TaskType, PeftModel
    import os, shutil

    base = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, target_modules=["query","value"])
    peft_model = get_peft_model(base, config)

    save_dir = "./lora_adapter_test"
    peft_model.save_pretrained(save_dir)
    size_kb = sum(os.path.getsize(os.path.join(save_dir, f))
                  for f in os.listdir(save_dir) if os.path.isfile(os.path.join(save_dir, f))) / 1024
    print(f"LoRA adapter size: {size_kb:.1f} KB (vs ~440MB for full BERT)")

    base2 = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
    loaded = PeftModel.from_pretrained(base2, save_dir)
    print("Adapter loaded successfully!")
    shutil.rmtree(save_dir)
except ImportError:
    print("Install: pip install peft transformers")

---
# Section 13: Fine-Tuning a Sentence Transformer

**Pipeline:** Load base model -> Prepare pairs -> MNRL loss -> Train -> Evaluate on STS.

### 13a-13d. Full Sentence Transformer Fine-Tuning Pipeline

In [ ]:
try:
    from sentence_transformers import SentenceTransformer, InputExample, losses, evaluation
    from torch.utils.data import DataLoader

    model = SentenceTransformer('all-MiniLM-L6-v2')

    train_examples = [
        InputExample(texts=["How to learn Python?", "Best resources for learning Python"]),
        InputExample(texts=["What is ML?", "Introduction to machine learning concepts"]),
        InputExample(texts=["Weather in NYC", "Current temperature and forecast"]),
        InputExample(texts=["Best restaurants", "Top rated dining places nearby"]),
    ] * 20

    train_dl = DataLoader(train_examples, shuffle=True, batch_size=8)
    train_loss = losses.MultipleNegativesRankingLoss(model)

    eval_s1 = ["Python programming", "ML basics", "Cooking recipes"]
    eval_s2 = ["Python coding tutorial", "ML fundamentals", "How to cook"]
    eval_scores = [0.9, 0.85, 0.8]
    evaluator = evaluation.EmbeddingSimilarityEvaluator(eval_s1, eval_s2, eval_scores)

    score_before = evaluator(model)
    print(f"Before training: {score_before:.4f}")

    model.fit(train_objectives=[(train_dl, train_loss)], epochs=1, warmup_steps=5,
              show_progress_bar=True)

    score_after = evaluator(model)
    print(f"After training: {score_after:.4f}")
except ImportError:
    print("Install: pip install sentence-transformers")

### 13e-13f. Evaluate and Save Fine-Tuned Model

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np, shutil

    model = SentenceTransformer('all-MiniLM-L6-v2')
    queries = ["How to learn programming?", "Best cooking recipes"]
    docs = ["Programming tutorials", "Advanced coding", "Food preparation guide", "Sports news"]

    sim = cosine_similarity(model.encode(queries), model.encode(docs))
    for i, q in enumerate(queries):
        ranking = np.argsort(-sim[i])
        print(f"Query: '{q}'")
        for j in ranking:
            print(f"  [{sim[i,j]:.3f}] {docs[j]}")
        print()

    model.save("./my_st_model")
    loaded = SentenceTransformer("./my_st_model")
    print(f"Saved and loaded, dim={loaded.get_sentence_embedding_dimension()}")
    shutil.rmtree("./my_st_model")
except ImportError:
    print("Install: pip install sentence-transformers")

---
# Section 14: RAG Pipeline (Retrieval-Augmented Generation)

RAG = Retrieve relevant context + Generate answer conditioned on it.

**Pipeline:** Query -> Embed -> Search index -> (Optional: Rerank) -> Construct prompt -> Generate.

**Interview tip:** Know chunking strategies, embedding choices, retrieval vs reranking, prompt engineering for RAG, and faithfulness evaluation.

### 14a. Load and Chunk Documents (Recursive Character Splitter)

In [ ]:
class RecursiveCharacterSplitter:
    """Split text into chunks with overlap.
    Tries to split on paragraphs, then sentences, then words, then chars.
    """
    def __init__(self, chunk_size=500, overlap=50,
                 separators=None):
        self.chunk_size = chunk_size
        self.overlap = overlap
        self.separators = separators or ["\n\n", "\n", ". ", " ", ""]

    def split(self, text):
        chunks = []
        self._split_recursive(text, self.separators, chunks)
        return chunks

    def _split_recursive(self, text, separators, chunks):
        if len(text) <= self.chunk_size:
            if text.strip():
                chunks.append(text.strip())
            return

        sep = separators[0]
        remaining_seps = separators[1:] if len(separators) > 1 else [""]

        if sep == "":
            # Character-level split
            for i in range(0, len(text), self.chunk_size - self.overlap):
                chunk = text[i:i + self.chunk_size]
                if chunk.strip():
                    chunks.append(chunk.strip())
            return

        parts = text.split(sep)
        current = ""
        for part in parts:
            if len(current) + len(part) + len(sep) > self.chunk_size:
                if current.strip():
                    chunks.append(current.strip())
                # Overlap: keep end of previous chunk
                if self.overlap > 0 and current:
                    current = current[-self.overlap:] + sep + part
                else:
                    current = part
            else:
                current = current + sep + part if current else part

        if current.strip():
            if len(current) > self.chunk_size and remaining_seps:
                self._split_recursive(current, remaining_seps, chunks)
            else:
                chunks.append(current.strip())

# Test
document = """
Machine learning is a subset of artificial intelligence that focuses on building systems
that learn from data. There are three main types of machine learning: supervised learning,
unsupervised learning, and reinforcement learning.

Supervised learning uses labeled data to train models. Common algorithms include linear
regression, decision trees, and neural networks. The model learns to map inputs to outputs
based on example input-output pairs.

Unsupervised learning finds patterns in unlabeled data. Clustering and dimensionality
reduction are common tasks. K-means, DBSCAN, and PCA are popular algorithms.

Reinforcement learning trains agents to make decisions by maximizing cumulative reward.
It is used in game playing, robotics, and recommendation systems.
"""

splitter = RecursiveCharacterSplitter(chunk_size=200, overlap=30)
chunks = splitter.split(document)
print(f"Document length: {len(document)} chars")
print(f"Number of chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i} ({len(chunk)} chars):\n  {chunk[:100]}...")

### 14b-14c. Embed Chunks and Build FAISS Index

In [ ]:
import numpy as np

# Use synthetic embeddings for demo (replace with real model in production)
np.random.seed(42)
chunks = [
    "Machine learning is a subset of AI that learns from data.",
    "Supervised learning uses labeled data. Common algos: linear regression, decision trees.",
    "Unsupervised learning finds patterns in unlabeled data. K-means, PCA.",
    "Reinforcement learning trains agents to maximize reward.",
    "Deep learning uses neural networks with many layers.",
    "Transfer learning reuses pretrained models for new tasks.",
    "NLP is a field focused on understanding human language.",
    "Computer vision processes and analyzes images and video.",
]

# Simulate embeddings (in practice: model.encode(chunks))
dim = 128
chunk_embeddings = np.random.randn(len(chunks), dim).astype('float32')
# Normalize for cosine similarity
norms = np.linalg.norm(chunk_embeddings, axis=1, keepdims=True)
chunk_embeddings = chunk_embeddings / norms

try:
    import faiss
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embeddings)
    print(f"FAISS index built: {index.ntotal} chunks, dim={dim}")
except ImportError:
    print("FAISS not available, using numpy for search")
    index = None

### 14d. Retrieval Function

In [ ]:
import numpy as np

def retrieve(query_text, chunks, chunk_embeddings, index=None, top_k=3):
    """Embed query, search index, return top-k chunks."""
    # In practice: query_emb = model.encode([query_text])
    np.random.seed(hash(query_text) % 2**32)
    query_emb = np.random.randn(1, chunk_embeddings.shape[1]).astype('float32')
    query_emb = query_emb / np.linalg.norm(query_emb)

    if index is not None:
        try:
            scores, indices = index.search(query_emb, top_k)
            return [(chunks[i], float(scores[0][j])) for j, i in enumerate(indices[0])]
        except:
            pass

    # Fallback: numpy search
    scores = (chunk_embeddings @ query_emb.T).squeeze()
    top_idx = np.argsort(-scores)[:top_k]
    return [(chunks[i], float(scores[i])) for i in top_idx]

results = retrieve("What is supervised learning?", chunks, chunk_embeddings, index)
print("Retrieved chunks:")
for chunk, score in results:
    print(f"  [{score:.4f}] {chunk}")

### 14e. Prompt Template for RAG

In [ ]:
def build_rag_prompt(query, retrieved_chunks, system_prompt=None):
    """Build a RAG prompt with retrieved context."""
    if system_prompt is None:
        system_prompt = (
            "You are a helpful assistant. Answer the question based ONLY on the provided context. "
            "If the context doesn't contain the answer, say 'I don't have enough information.'"
        )

    context = "\n\n".join([f"[{i+1}] {chunk}" for i, (chunk, _) in enumerate(retrieved_chunks)])

    prompt = f"""System: {system_prompt}

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# Build prompt
query = "What is supervised learning?"
retrieved = retrieve(query, chunks, chunk_embeddings, index)
prompt = build_rag_prompt(query, retrieved)
print(prompt)

### 14f. Generate Answer (API Call Placeholder)

In [ ]:
def generate_answer(prompt, model_name="gpt-3.5-turbo"):
    """Placeholder for LLM generation.
    In production, call OpenAI API, local model, or vLLM.
    """
    # Placeholder response
    print(f"[Would call {model_name} with prompt of {len(prompt)} chars]")
    return "Based on the context, supervised learning uses labeled data to train models..."

    # --- Real OpenAI API call ---
    # import openai
    # response = openai.ChatCompletion.create(
    #     model=model_name,
    #     messages=[{"role": "user", "content": prompt}],
    #     temperature=0.0,
    #     max_tokens=200,
    # )
    # return response.choices[0].message.content

answer = generate_answer(prompt)
print(f"Answer: {answer}")

### 14g. Full RAG Pipeline: Query -> Retrieve -> Rerank -> Generate

In [ ]:
def rag_pipeline(query, chunks, chunk_embeddings, index=None,
                 retrieval_top_k=5, rerank_top_k=3):
    """Complete RAG pipeline."""
    print(f"Query: '{query}'")

    # Step 1: Retrieve
    candidates = retrieve(query, chunks, chunk_embeddings, index, top_k=retrieval_top_k)
    print(f"\n1. Retrieved {len(candidates)} candidates")

    # Step 2: Rerank (simulated — in practice use cross-encoder)
    # Sort by original score as placeholder
    reranked = sorted(candidates, key=lambda x: -x[1])[:rerank_top_k]
    print(f"2. Reranked to top {rerank_top_k}")

    # Step 3: Build prompt
    prompt = build_rag_prompt(query, reranked)
    print(f"3. Built prompt ({len(prompt)} chars)")

    # Step 4: Generate
    answer = generate_answer(prompt)
    print(f"4. Generated answer")

    return {
        "query": query,
        "retrieved": candidates,
        "reranked": reranked,
        "prompt": prompt,
        "answer": answer,
    }

result = rag_pipeline("What is reinforcement learning?", chunks, chunk_embeddings, index)

### 14h. Evaluate with Simple Faithfulness Check

In [ ]:
def simple_faithfulness_check(answer, context_chunks):
    """Simple faithfulness check: what fraction of answer words appear in context?
    This is a very basic proxy. In practice, use NLI models or LLM-as-judge.
    """
    answer_words = set(answer.lower().split())
    context_words = set()
    for chunk, _ in context_chunks:
        context_words.update(chunk.lower().split())

    if not answer_words:
        return 0.0

    overlap = answer_words & context_words
    faithfulness = len(overlap) / len(answer_words)
    unsupported = answer_words - context_words

    return {
        "faithfulness_score": faithfulness,
        "supported_words": len(overlap),
        "total_words": len(answer_words),
        "unsupported_words": list(unsupported)[:10],
    }

answer = "Supervised learning uses labeled data to train models like linear regression and neural networks"
context = [("Supervised learning uses labeled data. Common algos: linear regression, decision trees.", 0.9)]

result = simple_faithfulness_check(answer, context)
print(f"Faithfulness: {result['faithfulness_score']:.2f}")
print(f"Supported: {result['supported_words']}/{result['total_words']}")
print(f"Unsupported words: {result['unsupported_words']}")

---
# Section 15: Supervised Fine-Tuning (SFT) with HuggingFace

SFT teaches a pretrained LLM to follow instructions by training on (instruction, response) pairs.

**Interview tip:** Know Alpaca format, chat templates, SFTTrainer from trl, and how LoRA makes SFT practical for large models.

### 15a-15b. Load Model and Prepare Dataset

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    # Use GPT-2 (small, no GPU needed)
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Set pad token (GPT-2 doesn't have one by default)
    tokenizer.pad_token = tokenizer.eos_token

    print(f"Model: {model_name}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Vocab size: {tokenizer.vocab_size}")

    # Alpaca-format instruction tuning data
    train_data = [
        {"instruction": "What is machine learning?",
         "response": "Machine learning is a subset of AI where systems learn from data."},
        {"instruction": "Explain Python in one sentence.",
         "response": "Python is a high-level programming language known for readability."},
        {"instruction": "What is a neural network?",
         "response": "A neural network is a computing system inspired by biological neurons."},
    ]

    def format_alpaca(example):
        return f"### Instruction:\n{example['instruction']}\n\n### Response:\n{example['response']}"

    for ex in train_data:
        formatted = format_alpaca(ex)
        tokens = tokenizer.encode(formatted)
        print(f"\n{formatted[:80]}... ({len(tokens)} tokens)")

except ImportError:
    print("Install: pip install transformers")

### 15c-15d. Tokenize and Set Up Training

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
    from torch.utils.data import Dataset
    import torch

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    class InstructDataset(Dataset):
        def __init__(self, data, tokenizer, max_len=128):
            self.encodings = []
            for ex in data:
                text = f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['response']}{tokenizer.eos_token}"
                enc = tokenizer(text, truncation=True, max_length=max_len,
                               padding="max_length", return_tensors="pt")
                # For causal LM, labels = input_ids (shifted internally by the model)
                enc["labels"] = enc["input_ids"].clone()
                # Mask padding in labels
                enc["labels"][enc["attention_mask"] == 0] = -100
                self.encodings.append({k: v.squeeze(0) for k, v in enc.items()})

        def __len__(self):
            return len(self.encodings)

        def __getitem__(self, idx):
            return self.encodings[idx]

    train_data = [
        {"instruction": "What is machine learning?",
         "response": "Machine learning is a subset of AI."},
        {"instruction": "Explain Python.",
         "response": "Python is a programming language."},
    ] * 10  # repeat for demo

    dataset = InstructDataset(train_data, tokenizer)
    print(f"Dataset size: {len(dataset)}")
    print(f"Sample keys: {list(dataset[0].keys())}")
    print(f"Input shape: {dataset[0]['input_ids'].shape}")

    # Training arguments
    training_args = TrainingArguments(
        output_dir="./sft_output",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        learning_rate=5e-5,
        logging_steps=5,
        save_strategy="no",  # don't save for demo
        report_to="none",
    )
    print(f"\nTraining args configured: {training_args.num_train_epochs} epoch(s)")

except ImportError:
    print("Install: pip install transformers")

### 15e-15f. Train and Compare Before/After Generation

In [ ]:
try:
    from transformers import AutoModelForCausalLM, Trainer
    import torch

    model = AutoModelForCausalLM.from_pretrained("gpt2")

    def generate_text(model, tokenizer, prompt, max_new_tokens=50):
        inputs = tokenizer(prompt, return_tensors="pt")
        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy for reproducibility
                pad_token_id=tokenizer.eos_token_id,
            )
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    prompt = "### Instruction:\nWhat is machine learning?\n\n### Response:\n"
    print("BEFORE SFT:")
    print(generate_text(model, tokenizer, prompt, max_new_tokens=30))

    # Short training (1 epoch on tiny data)
    trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
    trainer.train()

    print("\nAFTER SFT:")
    print(generate_text(model, tokenizer, prompt, max_new_tokens=30))

    # Cleanup
    import shutil
    if __import__('os').path.exists("./sft_output"):
        shutil.rmtree("./sft_output")

except ImportError:
    print("Install: pip install transformers")
except Exception as e:
    print(f"Training skipped: {e}")

### 15g. Apply LoRA for Parameter-Efficient SFT

In [ ]:
try:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import get_peft_model, LoraConfig, TaskType

    model = AutoModelForCausalLM.from_pretrained("gpt2")
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

    # LoRA config for causal LM
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["c_attn"],  # GPT-2 attention projection
    )

    peft_model = get_peft_model(model, lora_config)
    peft_model.print_trainable_parameters()
    # Expected: ~0.5-1% trainable

except ImportError:
    print("Install: pip install peft transformers")

---
# Section 16: Inference Optimization

Techniques to make LLM inference faster and cheaper.

**Interview tip:** Know quantization levels (FP16, INT8, INT4), KV cache, and decoding strategies.

### 16a-16b. Quantization Concepts

In [ ]:
# Quantization reduces model size and speeds up inference by using lower precision.
# Real quantization requires bitsandbytes or GPTQ. Here we show the concepts.

import torch
import numpy as np

def simulate_quantization(tensor, bits=8):
    """Simulate quantization: map float values to integer range.
    Real INT8: values mapped to [-128, 127]
    Real INT4: values mapped to [-8, 7]
    """
    qmin = -(2 ** (bits - 1))
    qmax = 2 ** (bits - 1) - 1

    # Compute scale and zero point (symmetric quantization)
    abs_max = tensor.abs().max()
    scale = abs_max / qmax

    # Quantize
    quantized = torch.round(tensor / scale).clamp(qmin, qmax).to(torch.int8)

    # Dequantize
    dequantized = quantized.float() * scale

    # Measure error
    error = (tensor - dequantized).abs()

    return {
        "scale": scale.item(),
        "quantized_dtype_size": bits / 8,
        "max_error": error.max().item(),
        "mean_error": error.mean().item(),
        "compression_ratio": 32 / bits,
    }

# Test on random weights
weights = torch.randn(1000, 1000)
print(f"Original: {weights.element_size()} bytes/param, {weights.numel() * 4 / 1e6:.1f} MB")

for bits in [16, 8, 4]:
    result = simulate_quantization(weights, bits)
    size_mb = weights.numel() * result["quantized_dtype_size"] / 1e6
    print(f"\nINT{bits}: {result['compression_ratio']:.0f}x compression, {size_mb:.1f} MB")
    print(f"  Max error: {result['max_error']:.6f}, Mean error: {result['mean_error']:.6f}")

### 16c. Measure Inference Speed: FP32 vs FP16

In [ ]:
import torch
import time

def benchmark_matmul(size=1024, dtype=torch.float32, iterations=100):
    a = torch.randn(size, size, dtype=dtype)
    b = torch.randn(size, size, dtype=dtype)
    # Warmup
    for _ in range(10):
        _ = a @ b
    # Benchmark
    start = time.time()
    for _ in range(iterations):
        _ = a @ b
    elapsed = (time.time() - start) / iterations * 1000
    return elapsed

print(f"{'Dtype':<10} {'ms/matmul':<12} {'Speedup':<10}")
print("-" * 32)
fp32_time = benchmark_matmul(dtype=torch.float32)
print(f"{'FP32':<10} {fp32_time:<12.2f} {'1.0x':<10}")

fp16_time = benchmark_matmul(dtype=torch.float16)
print(f"{'FP16':<10} {fp16_time:<12.2f} {fp32_time/fp16_time:.1f}x")

# Note: on CPU, FP16 may not be faster. On GPU, expect 2-4x speedup.
print("\nNote: FP16 speedup is mainly visible on GPU (CUDA cores / Tensor cores)")

### 16d. KV Cache Size Calculator

In [ ]:
def kv_cache_size(num_layers, d_model, max_seq_len, batch_size=1,
                  dtype_bytes=2, num_kv_heads=None, num_heads=32):
    """Calculate KV cache memory in GB.
    KV cache stores past keys and values to avoid recomputation.
    Size = 2 * num_layers * batch * seq_len * d_head * num_kv_heads * dtype_bytes
    """
    if num_kv_heads is None:
        num_kv_heads = num_heads
    d_head = d_model // num_heads
    total_bytes = 2 * num_layers * batch_size * max_seq_len * d_head * num_kv_heads * dtype_bytes
    return total_bytes / (1024 ** 3)

print("KV Cache Size (FP16):")
print(f"{'Model':<25} {'Seq 4K':>8} {'Seq 16K':>8} {'Seq 128K':>8}")
print("-" * 55)
for name, nl, dm, nh, nkv in [
    ("LLaMA-7B (MHA)", 32, 4096, 32, 32),
    ("LLaMA-7B (GQA-8)", 32, 4096, 32, 8),
    ("LLaMA-70B (GQA-8)", 80, 8192, 64, 8),
]:
    row = f"{name:<25}"
    for seq in [4096, 16384, 131072]:
        row += f" {kv_cache_size(nl, dm, seq, 1, 2, nkv, nh):>7.2f}G"
    print(row)

### 16e. Batch Inference with Padding and Attention Masks

In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    model = AutoModelForCausalLM.from_pretrained("gpt2")
    model.eval()

    # Pad token setup
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # left-pad for causal LM generation

    # Batch of different-length inputs
    prompts = [
        "The capital of France is",
        "Machine learning",
        "The quick brown fox jumps over the",
    ]

    # Tokenize with padding
    inputs = tokenizer(prompts, return_tensors="pt", padding=True)
    print(f"Input IDs shape: {inputs['input_ids'].shape}")
    print(f"Attention mask shape: {inputs['attention_mask'].shape}")
    print(f"Attention mask (1=real, 0=pad):\n{inputs['attention_mask']}")

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            pad_token_id=tokenizer.eos_token_id,
        )

    for i, (prompt, output) in enumerate(zip(prompts, outputs)):
        generated = tokenizer.decode(output, skip_special_tokens=True)
        print(f"\n[{i}] {generated}")

except ImportError:
    print("Install: pip install transformers")

### 16f. Greedy vs Sampling vs Top-K vs Top-P Decoding

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def decode_step(logits, strategy="greedy", temperature=1.0, top_k=0, top_p=0.0):
    """Single decoding step with different strategies.

    Strategies:
    - greedy: always pick highest probability token
    - sampling: sample from the full distribution
    - top-k: sample from top-k tokens only
    - top-p (nucleus): sample from smallest set of tokens whose cumulative prob >= p
    """
    logits = logits / temperature

    if strategy == "greedy":
        return logits.argmax(dim=-1)

    if top_k > 0:
        # Zero out everything except top-k
        values, _ = torch.topk(logits, top_k)
        min_val = values[:, -1:]
        logits = torch.where(logits < min_val, torch.tensor(float('-inf')), logits)

    if top_p > 0:
        # Nucleus sampling
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
        # Remove tokens with cumulative prob above threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        sorted_indices_to_remove[:, 1:] = sorted_indices_to_remove[:, :-1].clone()
        sorted_indices_to_remove[:, 0] = False
        # Scatter back
        indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
        logits = logits.masked_fill(indices_to_remove, float('-inf'))

    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

# Simulate logits for vocab_size=10
torch.manual_seed(42)
logits = torch.randn(1, 10)
probs = F.softmax(logits, dim=-1)

print(f"Probabilities: {probs[0].numpy().round(3)}")
print(f"Greedy:     token {decode_step(logits, 'greedy').item()}")

# Sample multiple times to show distribution
for strategy, kwargs in [
    ("top-k=3", {"top_k": 3}),
    ("top-p=0.9", {"top_p": 0.9}),
    ("temp=0.5", {"temperature": 0.5}),
    ("temp=2.0", {"temperature": 2.0}),
]:
    samples = [decode_step(logits, "sampling", **kwargs).item() for _ in range(20)]
    unique = set(samples)
    print(f"{strategy:<12}: sampled tokens {samples[:10]}... unique={len(unique)}")

---
# Section 17: CLIP (Multi-Modal)

CLIP learns to align images and text in a shared embedding space using contrastive learning.

**Key ideas:**
- Encode images and text into same vector space
- Zero-shot classification: compare image embedding to text embeddings of class names
- Image search: find images closest to a text query

**Interview tip:** Know CLIP's training objective (InfoNCE on image-text pairs), its zero-shot capabilities, and limitations.

### 17a-17c. Load CLIP, Encode, and Compute Similarity

In [ ]:
try:
    from transformers import CLIPProcessor, CLIPModel
    import torch
    import numpy as np

    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    # Since we may not have real images, demonstrate with text-text similarity
    # (CLIP's text encoder can still show the concept)
    texts = [
        "a photo of a cat",
        "a photo of a dog",
        "a diagram of a neural network",
        "a photo of a sunset at the beach",
    ]

    inputs = processor(text=texts, return_tensors="pt", padding=True)
    with torch.no_grad():
        text_features = model.get_text_features(**inputs)

    # Normalize
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    # Pairwise similarity
    sim = (text_features @ text_features.T).numpy()
    print("Text-Text similarity matrix (CLIP):")
    for i, t in enumerate(texts):
        print(f"  [{i}] {t}")
    print(np.round(sim, 3))

except ImportError:
    print("Install: pip install transformers")
    print("CLIP encodes images and text into the same embedding space.")
    print("Use case: zero-shot classification, image search, multimodal RAG.")

### 17d. Zero-Shot Classification (Concept)

In [ ]:
import torch
import torch.nn.functional as F

def zero_shot_classify(image_features, class_texts, text_features):
    """Zero-shot classification with CLIP.
    Compare image embedding to text embeddings of candidate class descriptions.
    """
    # Normalize
    image_features = F.normalize(image_features, dim=-1)
    text_features = F.normalize(text_features, dim=-1)

    # Compute similarity
    similarity = image_features @ text_features.T  # (num_images, num_classes)

    # Softmax to get probabilities
    probs = F.softmax(similarity * 100, dim=-1)  # CLIP uses logit_scale ~100

    return probs

# Simulate
num_classes = 5
class_names = ["cat", "dog", "car", "airplane", "flower"]

# Simulate image and text features (in practice from CLIP encoder)
torch.manual_seed(42)
image_feat = torch.randn(1, 512)  # one image
text_feats = torch.randn(num_classes, 512)  # one per class

probs = zero_shot_classify(image_feat, class_names, text_feats)
print("Zero-shot classification:")
for name, prob in zip(class_names, probs[0]):
    print(f"  {name}: {prob.item():.3f}")
print(f"Predicted: {class_names[probs[0].argmax()]}")

### 17e. Image Search by Text Query (Concept)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

def text_to_image_search(query_feat, image_feats, image_ids, top_k=3):
    """Search images by text query in CLIP embedding space."""
    query_feat = F.normalize(query_feat, dim=-1)
    image_feats = F.normalize(image_feats, dim=-1)

    scores = (query_feat @ image_feats.T).squeeze()
    top_idx = scores.argsort(descending=True)[:top_k]

    return [(image_ids[i], scores[i].item()) for i in top_idx]

# Simulate image database
image_ids = [f"img_{i:04d}.jpg" for i in range(100)]
image_feats = torch.randn(100, 512)

# Text query
query_feat = torch.randn(1, 512)  # In practice: CLIP text encoder

results = text_to_image_search(query_feat, image_feats, image_ids, top_k=5)
print("Image search results:")
for img_id, score in results:
    print(f"  [{score:.4f}] {img_id}")

print("\nIn practice: encode query with CLIP text encoder, compare to precomputed image embeddings.")

---
# Section 18: Utility Functions

Commonly needed helper functions for LLM/ML engineering.

### 18a. Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    """RRF: combine multiple ranked lists.
    Score(d) = sum_r 1/(k + rank_r(d))
    Standard k=60 from the original paper."""
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])

r1 = [2, 0, 5, 1, 3]
r2 = [5, 2, 1, 0, 4]
fused = reciprocal_rank_fusion([r1, r2])
print("RRF fusion:", fused)

### 18b. Chunk Text with Overlap

In [ ]:
def chunk_text(text, chunk_size=500, overlap=50):
    """Simple character-level chunking with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

text = "A" * 1200
chunks = chunk_text(text, chunk_size=500, overlap=50)
print(f"Text length: {len(text)}")
print(f"Chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"  Chunk {i}: {len(c)} chars")

### 18c. Batch Encode with Progress Bar

In [ ]:
from tqdm import tqdm
import numpy as np

def batch_encode(texts, encode_fn, batch_size=32, show_progress=True):
    """Encode texts in batches with progress bar."""
    all_embeddings = []
    iterator = range(0, len(texts), batch_size)
    if show_progress:
        iterator = tqdm(iterator, desc="Encoding")

    for i in iterator:
        batch = texts[i:i + batch_size]
        # In practice: embeddings = model.encode(batch)
        embeddings = np.random.randn(len(batch), 128)  # placeholder
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

texts = [f"Document {i}" for i in range(200)]
embs = batch_encode(texts, None, batch_size=32)
print(f"Encoded {len(texts)} texts -> {embs.shape}")

### 18d. GPU Memory Usage Monitoring

In [ ]:
import torch

def gpu_memory_stats():
    """Print GPU memory usage if CUDA is available."""
    if not torch.cuda.is_available():
        print("No GPU available. CUDA not detected.")
        print("On a GPU system, this would show:")
        print("  Allocated: X.XX GB")
        print("  Reserved:  X.XX GB")
        print("  Free:      X.XX GB")
        return

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total:.2f} GB")
    print(f"  Free:      {total - allocated:.2f} GB")

gpu_memory_stats()

### 18e. Simple LRU Cache for Embeddings

In [ ]:
from collections import OrderedDict
import numpy as np

class EmbeddingCache:
    """LRU cache for embeddings to avoid recomputation."""
    def __init__(self, max_size=10000):
        self.cache = OrderedDict()
        self.max_size = max_size
        self.hits = 0
        self.misses = 0

    def get(self, key):
        if key in self.cache:
            self.cache.move_to_end(key)
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def put(self, key, embedding):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = embedding
        if len(self.cache) > self.max_size:
            self.cache.popitem(last=False)

    def get_or_compute(self, key, compute_fn):
        result = self.get(key)
        if result is None:
            result = compute_fn(key)
            self.put(key, result)
        return result

    @property
    def hit_rate(self):
        total = self.hits + self.misses
        return self.hits / total if total > 0 else 0

cache = EmbeddingCache(max_size=100)

# Simulate encoding with cache
def fake_encode(text):
    return np.random.randn(128)

texts = ["hello", "world", "hello", "foo", "hello", "world", "bar"]
for text in texts:
    emb = cache.get_or_compute(text, fake_encode)

print(f"Hits: {cache.hits}, Misses: {cache.misses}")
print(f"Hit rate: {cache.hit_rate:.1%}")
print(f"Cache size: {len(cache.cache)}")

### 18f. Timer Decorator for Benchmarking

In [ ]:
import time
import functools

def timer(func):
    """Decorator that prints execution time."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__}: {elapsed:.4f}s")
        return result
    return wrapper

class Timer:
    """Context manager for timing code blocks."""
    def __init__(self, name=""):
        self.name = name

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        print(f"{self.name}: {self.elapsed:.4f}s")

# Usage as decorator
@timer
def slow_function():
    time.sleep(0.1)
    return 42

result = slow_function()

# Usage as context manager
with Timer("matrix multiply"):
    import numpy as np
    a = np.random.randn(1000, 1000)
    b = np.random.randn(1000, 1000)
    c = a @ b

---
# Summary

This notebook covers the essential building blocks for LLM engineering:

| Section | Topics |
|---------|--------|
| 1. Similarity Metrics | Cosine, dot product, Euclidean |
| 2. Tokenization | BPE from scratch, HuggingFace tokenizers |
| 3. Attention | Scaled dot-product, causal, multi-head |
| 4. Transformer | LayerNorm, RMSNorm, FFN, RoPE, full encoder/decoder |
| 5. Model Inspection | Parameter counting, memory estimation, KV cache |
| 6. Embeddings | BERT pooling, sentence-transformers, anisotropy |
| 7. FAISS | Flat, IVF, HNSW, PQ indices |
| 8. BM25 | From scratch, hybrid search with RRF |
| 9. Cross-Encoder | Reranking pipeline, MRR/NDCG evaluation |
| 10. Evaluation | NDCG, MRR, MAP, Recall, Precision, BERTScore |
| 11. Loss Functions | CE, contrastive, triplet, InfoNCE, MNRL, DPO |
| 12. LoRA | From scratch, merge, PEFT library |
| 13. Fine-Tuning ST | MNRL training, evaluation |
| 14. RAG Pipeline | Chunking, retrieval, reranking, generation |
| 15. SFT | Instruction tuning, LoRA SFT |
| 16. Inference | Quantization, KV cache, decoding strategies |
| 17. CLIP | Multi-modal embeddings, zero-shot classification |
| 18. Utilities | RRF, chunking, caching, benchmarking |

**Good luck with your interview!**